# Verification: shells (2026-10-14) and mass, center of mass and work (2026-10-19 / 2026-10-20)

This notebook supports two updates of `study-guides/calc2_study_guide_1.1-3.3.html` (the filename is historical; the guide now runs through §3.9):
- **2026-10-14**, `materials/notes/MATH1342_Fall 2026_Section 12_0.pdf`: the **§3.5 cylindrical shells** topic (S), three new washer problems about other axes (V-p11–V-p13) and three mixed items (Mixed 27–29). That update was never given its own notebook, so it is verified here.
- **2026-10-19**, `materials/notes/3.7_3.9.pdf`: **§3.7 mass and density** (K), **§3.8 center of mass** (G), **§3.9 work** (W), plus diagnostic, mixed-practice (Mixed 30–35) and final-check (items 31–42) items. It was finished on 2026-10-20.

Method, as in `verify_arclength_volume.ipynb`:
1. Every answer was derived **by hand** first (the `claim` column).
2. **SymPy** recomputes each one **from the problem statement**, and an **independent numerical route** checks it. Where possible the route is a *different method*: shells are checked by slicing the other way (horizontal strips), centers of mass by double integrals over the region (no midpoint formula), tank and rope work by (weight) × (rise of the center of mass), and leaky buckets in the time domain.
3. Every targeted wrong-answer value is recomputed from the mistake its message describes. Numbers quoted in explanations, tryouts, spot-the-error items, figures and "What to notice" lists are checked against the page source.
4. The guide's **own grader** (extracted from `app.js`) is run on every expression key (`js_eval.js`) and every numeric key (`js_eval_num.js`, new): references accepted, wrong answers get their own message, units handled.
5. **Wolfram Language** results, computed from the problem statements, are read verbatim from `wolfram_shells_mass_work_checks.txt` (batches 1–4 run on 2026-10-19, 5a–5b on 2026-10-20) and compared.
6. The **built HTML** is audited against `guide_state_before_2026-10-14.json` and `guide_state_before_2026-10-19.json`: every new answer verified, every older answer unchanged, nothing lost.

Notation: "3.7–3.9 notes p. N" is `3.7_3.9.pdf`; "MATH1342 notes p. N" is `MATH1342_Fall 2026_Section 12_0.pdf`. Guide topic letters: S shells, K mass, G center of mass, W work.

In [1]:
import sympy as sp, mpmath as mp, numpy as np, json, re, subprocess, pathlib, html as _html, warnings
from scipy import integrate as si, optimize as so
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
warnings.filterwarnings('ignore', category=si.IntegrationWarning)
mp.mp.dps = 30
x, y, t, u = sp.symbols('x y t u', real=True)
c, k = sp.symbols('c k', positive=True)
pi, E, sqrt, exp, log, sin, R, I = sp.pi, sp.E, sp.sqrt, sp.exp, sp.log, sp.sin, sp.Rational, sp.integrate
ROOT = pathlib.Path(r"C:\Users\chris\MathStudy")
BUILD = ROOT / "study-guides" / "_build" / "calc2_ftc_improper_numerical_displacement"
GUIDE = ROOT / "study-guides" / "calc2_study_guide_1.1-3.3.html"
VER = ROOT / "study-guides" / "verification"
PARTS = "\n".join(p.read_text(encoding='utf-8') for p in sorted((BUILD / 'parts').glob('*.html')))
ROWS, VERIFIED = [], {}

def fnum(e): return float(sp.N(sp.sympify(e), 30))
def check(vid, claim, sym_val, numeric_val=None, ntol=1e-9, note=''):
    """claim = hand-derived exact answer; sym_val = SymPy from the problem statement; numeric_val = independent numerical route."""
    c_, s_ = fnum(claim), fnum(sym_val)
    n_ = None if numeric_val is None else float(numeric_val)
    near = lambda a_, b_, r_: abs(a_ - b_) <= r_ * max(1.0, abs(a_), abs(b_))
    ok = near(c_, s_, 1e-13) and (n_ is None or near(c_, n_, ntol))
    ROWS.append(dict(id=vid, claim=str(claim), sympy=str(sym_val)[:60], numeric=n_, ok=ok, note=note))
    VERIFIED[vid] = c_
    print(f"{'OK      ' if ok else 'MISMATCH'} {vid:<22} claim={str(claim):<30.30} value={c_:<22.15g} numeric={'' if n_ is None else f'{n_:.13g}':<18} {note}")
    return ok

# ---- symbolic set-ups
def shell_sym(r, h, a, b): return I(2 * pi * r * h, (x, a, b))
def wash_sym(Ro, ri, a, b): return I(pi * (Ro**2 - ri**2), (x, a, b))
def wire_sym(d, a, b): M = I(d, (x, a, b)); return M, I(x * d, (x, a, b)) / M
def plate_sym(d, f, g, a, b):
    M = I(d * (f - g), (x, a, b)); return M, I(x * d * (f - g), (x, a, b)) / M, I((f + g) / 2 * d * (f - g), (x, a, b)) / M
def pick_root(eq, var, a, b):
    out = []
    for s_ in sp.solve(eq, var):
        z = complex(sp.N(s_, 30))
        if abs(z.imag) < 1e-15 and a - 1e-12 <= z.real <= b + 1e-12: out.append(s_)
    assert len(out) == 1, (eq, out); return out[0]
def mid_sym(d, a, b):
    M = I(d, (x, a, b)); return pick_root(sp.Eq(I(d, (x, a, t)), M / 2), t, a, b)

# ---- independent numerical routes
mq = lambda f, pts: mp.quad(f, pts)
def mid_num(lam, a, b):
    """Mass midpoint by a bracketing root-finder on [a, b] (relative scale: the equation is divided by M)."""
    M = mq(lam, [a, b]); return mp.findroot(lambda s_: (mq(lam, [a, s_]) - M / 2) / M, (a, b), solver='illinois')
def xspan(up, lo, a, b, yv, N=4000):
    """The x-interval of the region {lo(x) <= y <= up(x)} at height yv (all regions here are x-simple AND y-simple)."""
    xs = np.linspace(a, b, N + 1); f = np.minimum(up(xs) - yv, yv - lo(xs)); ins = np.nonzero(f >= 0)[0]
    if len(ins) == 0: return None
    g = lambda s_: min(up(s_) - yv, yv - lo(s_)); i0, i1 = ins[0], ins[-1]
    xL = a if i0 == 0 else so.brentq(g, xs[i0 - 1], xs[i0], xtol=1e-15, rtol=1e-15)
    xR = b if i1 == N else so.brentq(g, xs[i1], xs[i1 + 1], xtol=1e-15, rtol=1e-15)
    return xL, xR
def vol_slices_y(up, lo, a, b, ylo, yhi, axis_x=None, axis_y=None, brk=()):
    """Slice the region HORIZONTALLY instead. About a vertical axis x = axis_x a horizontal strip spins into a washer,
    pi(R^2 - r^2) dy; about a horizontal axis y = axis_y it spins into a shell, 2 pi |y - axis_y| (xR - xL) dy."""
    def f(yv):
        s_ = xspan(up, lo, a, b, yv)
        if s_ is None: return 0.0
        if axis_x is not None: d = sorted([abs(s_[0] - axis_x), abs(s_[1] - axis_x)]); return np.pi * (d[1]**2 - d[0]**2)
        return 2 * np.pi * abs(yv - axis_y) * (s_[1] - s_[0])
    pts = sorted(p_ for p_ in brk if ylo < p_ < yhi)
    return si.quad(f, ylo, yhi, points=pts or None, limit=500, epsabs=1e-12, epsrel=1e-12)[0]
def plate_num(d, up, lo, a, b):
    """Mass and center of mass by DOUBLE integrals over the region: no strip midpoint formula involved."""
    o = dict(epsabs=1e-13, epsrel=1e-12)
    M = si.dblquad(lambda yy, xx: d(xx), a, b, lo, up, **o)[0]
    Mx = si.dblquad(lambda yy, xx: xx * d(xx), a, b, lo, up, **o)[0]
    My = si.dblquad(lambda yy, xx: yy * d(xx), a, b, lo, up, **o)[0]
    return M, Mx / M, My / M
def tank_cm(w, L, D):
    """Slicing problems: W = (total weight) x (rise of the center of mass), w = weight per unit height on 0 <= y <= L, destination D."""
    wt = mq(w, [0, L]); yb = mq(lambda s_: s_ * w(s_), [0, L]) / wt; return wt * (D - yb)
print("helpers ready")

helpers ready


## 1. The lectures' own worked answers

`3.7_3.9.pdf` (§3.7–§3.9). The MATH1342 handout's shell and washer set-ups (pp. 5, 7–9) are the guide's S-guided, S-p1–S-p4, V-p12 and V-p13, checked in section 2.

In [2]:
print("== 3.7_3.9.pdf: the professor's worked answers ==")
check('n37-p2-M', 8, I(3*x**2, (x, 0, 2)), mq(lambda s: 3*s**2, [0, 2]), note='p. 2 writes 8')
check('n37-p2-xm', sp.cbrt(4), mid_sym(3*x**2, 0, 2), mid_num(lambda s: 3*s**2, 0, 2), note='p. 2 writes cbrt 4')
check('n37-p4-M-setup', 8*(5*log(3) - 4)/(3*log(3)**3), plate_sym(3**x, 2 - x**2, x**2, -1, 1)[0],
      plate_num(lambda s: 3.0**s, lambda s: 2 - s*s, lambda s: s*s, -1, 1)[0], ntol=1e-8, note='p. 4 is set up only; value for checking')
check('n37-p5-M', R(4, 3), plate_sym(x, x - x**2, -x, 0, 2)[0], plate_num(lambda s: s, lambda s: s - s*s, lambda s: -s, 0, 2)[0], ntol=1e-8, note='p. 5 writes 4/3')
check('n37-p8-M', 42, I(4*x**2 + 2, (x, 0, 3)), mq(lambda s: 4*s**2 + 2, [0, 3]))
check('n37-p8-Mx', 90, I(x*(4*x**2 + 2), (x, 0, 3)), mq(lambda s: s*(4*s**2 + 2), [0, 3]))
check('n37-p8-xbar', R(15, 7), wire_sym(4*x**2 + 2, 0, 3)[1], mq(lambda s: s*(4*s**2 + 2), [0, 3]) / mq(lambda s: 4*s**2 + 2, [0, 3]), note='p. 8 writes 90/42')
d9 = 2*exp(3*x); n9 = lambda s: 2*mp.e**(3*s)
check('n37-p9-M', R(2, 3)*(E**12 - E**9), I(d9, (x, 3, 4)), mq(n9, [3, 4]))
check('n37-p9-xm', log((E**12 + E**9)/2)/3, mid_sym(d9, 3, 4), mid_num(n9, 3, 4))
check('n37-p9-Mx-true', R(2, 9)*(11*E**12 - 8*E**9), I(x*d9, (x, 3, 4)), mq(lambda s: s*n9(s), [3, 4]), note='p. 9 writes (2/3)(11e^12 - 8e^9): see below')
check('n37-p9-xbar', (11*E**12 - 8*E**9)/(3*(E**12 - E**9)), wire_sym(d9, 3, 4)[1], mq(lambda s: s*n9(s), [3, 4]) / mq(n9, [3, 4]), note="p. 9's final answer is right")
F9 = R(2, 3)*x*exp(3*x) - R(2, 9)*exp(3*x)
print("   p. 9 antiderivative (2/3)x e^(3x) - (2/9)e^(3x): derivative minus integrand =", sp.simplify(sp.diff(F9, x) - d9*x), "(correct)")
AS_WRITTEN = R(2, 3)*(11*E**12 - 8*E**9)
print("   p. 9 moment as written / true moment =", sp.nsimplify(sp.simplify(AS_WRITTEN / I(x*d9, (x, 3, 4)))), " -> the written coefficient 2/3 should be 2/9 (a slip)")
g12 = plate_sym(x**2, 2/x**2, 0*x, 1, 2); n12 = plate_num(lambda s: s*s, lambda s: 2/(s*s), lambda s: 0.0, 1, 2)
check('n37-p12-M', 2, g12[0], n12[0], ntol=1e-8); check('n37-p12-Mx', 3, g12[0]*g12[1], n12[0]*n12[1], ntol=1e-8); check('n37-p12-My', 1, g12[0]*g12[2], n12[0]*n12[2], ntol=1e-8)
g13 = plate_sym(2**x, 2*x + 3, x**2, -1, 3); n13 = plate_num(lambda s: 2.0**s, lambda s: 2*s + 3, lambda s: s*s, -1, 3)
check('n37-p13-M', (34*log(2) - 15)/log(2)**3, g13[0], n13[0], ntol=1e-8, note='p. 13 is set up only; parts twice')
check('n37-p13-xbar', -3/log(2) + (94*log(2) - 15)/(34*log(2) - 15), g13[1], n13[1], ntol=1e-8, note='numeric: double integral')
check('n37-p13-ybar', (-90 + log(2)*(294 + log(2)*(-399 + 290*log(2))))/(log(2)**2*(34*log(2) - 15)), g13[2], n13[2], ntol=1e-8, note='numeric: double integral, no midpoint formula')
g98 = R(98, 10)
check('n37-p15-W', 5000, (2*50)*50, None, note='constant force: W = Fd')
check('n37-p16-W', 2500, I(2*(50 - y), (y, 0, 50)), tank_cm(lambda s: 2, 50, 50), note='numeric: weight x rise of the center of mass')
check('n37-p17-W', 20947500, I(1000*5**2*g98*(14 - y), (y, 0, 9)), tank_cm(lambda s: 1000*25*mp.mpf('9.8'), 9, 14))
check('n37-p18-W', 2940000*pi, I(1000*pi*(5*y/12)**2*g98*(12 - y), (y, 0, 12)), tank_cm(lambda s: 1000*mp.mpf('9.8')*mp.pi*(5*s/12)**2, 12, 12))
check('n37-p19-W', 74880000*pi, I(75*pi*((100 - y)/5)**2*(100 - y), (y, 0, 80)), tank_cm(lambda s: 75*mp.pi*((100 - s)/5)**2, 80, 100))
k21 = sp.solve(sp.Eq(k*R(4, 12), 10), k)[0]
check('n37-p21-k', 30, k21, 10/(4/12), note='4 in = 1/3 ft first')
check('n37-p21-W', R(15, 4), I(k21*x, (x, 0, R(6, 12))), mq(lambda s: 30*s, [0, 0.5]))
check('n37-p21-compress', 20, I(10*x, (x, 0, -2)), mq(lambda s: 10*s, [0, -2]))
check('n37-p22-k', 30, sp.solve(sp.Eq(I(k*x, (x, 0, 4)), 240), k)[0], 240/mq(lambda s: s, [0, 4]))
check('n37-p23-W', 2700, I(-R(1, 10)*y + 32, (y, 0, 100)), mq(lambda s: (32 - 0.2*s)*2, [0, 50]), note='numeric: in time, F(t) v dt')
xt = 4 - t**2; vt = sp.diff(xt, t)
check('n37-p24-W', 180, R(1, 2)*10*vt.subs(t, 3)**2 - R(1, 2)*10*vt.subs(t, 0)**2, mq(lambda s: 10*(-2)*(-2*s), [0, 3]), note="numeric: integral of F dx = m x''(t) x'(t) dt")

== 3.7_3.9.pdf: the professor's worked answers ==
OK       n37-p2-M               claim=8                              value=8                      numeric=8                  p. 2 writes 8
OK       n37-p2-xm              claim=2**(2/3)                       value=1.5874010519682        numeric=1.587401051968     p. 2 writes cbrt 4


OK       n37-p4-M-setup         claim=(-32 + 40*log(3))/(3*log(3)**3 value=3.00270768164087       numeric=3.002707681641     p. 4 is set up only; value for checking
OK       n37-p5-M               claim=4/3                            value=1.33333333333333       numeric=1.333333333333     p. 5 writes 4/3
OK       n37-p8-M               claim=42                             value=42                     numeric=42                 
OK       n37-p8-Mx              claim=90                             value=90                     numeric=90                 
OK       n37-p8-xbar            claim=15/7                           value=2.14285714285714       numeric=2.142857142857     p. 8 writes 90/42
OK       n37-p9-M               claim=-2*exp(9)/3 + 2*exp(12)/3      value=103101.138327619       numeric=103101.1383276     


OK       n37-p9-xm              claim=log(exp(9)/2 + exp(12)/2)/3    value=3.78514672367127       numeric=3.785146723671     
OK       n37-p9-Mx-true         claim=-16*exp(9)/9 + 22*exp(12)/9    value=383439.563152987       numeric=383439.563153      p. 9 writes (2/3)(11e^12 - 8e^9): see below
OK       n37-p9-xbar            claim=(-8*exp(9) + 11*exp(12))/(-3*e value=3.71906236315792       numeric=3.719062363158     p. 9's final answer is right
   p. 9 antiderivative (2/3)x e^(3x) - (2/9)e^(3x): derivative minus integrand = 0 (correct)
   p. 9 moment as written / true moment = 3  -> the written coefficient 2/3 should be 2/9 (a slip)
OK       n37-p12-M              claim=2                              value=2                      numeric=2                  
OK       n37-p12-Mx             claim=3                              value=3                      numeric=3                  
OK       n37-p12-My             claim=1                              value=1                      numeric=1

OK       n37-p13-M              claim=(-15 + 34*log(2))/log(2)**3    value=25.7248347468371       numeric=25.72483474684     p. 13 is set up only; parts twice
OK       n37-p13-xbar           claim=-3/log(2) + (-15 + 94*log(2))/ value=1.52645097403985       numeric=1.52645097404      numeric: double integral
OK       n37-p13-ybar           claim=(-90 + ((-399 + 290*log(2))*lo value=4.53388552904279       numeric=4.533885529043     numeric: double integral, no midpoint formula
OK       n37-p15-W              claim=5000                           value=5000                   numeric=                   constant force: W = Fd
OK       n37-p16-W              claim=2500                           value=2500                   numeric=2500               numeric: weight x rise of the center of mass
OK       n37-p17-W              claim=20947500                       value=20947500               numeric=20947500           
OK       n37-p18-W              claim=2940000*pi                     value=9

True

## 2. Every new numeric answer in the guide

2026-10-14 items first (washers about other axes, shells, Mixed 27–29), then the 2026-10-19 items (K, G, W, Mixed 30–35, final items 32–42). Shell volumes are checked numerically by **horizontal slicing** (washers in $y$); washer volumes about horizontal axes by **shells in $y$**; plate centers of mass by **double integrals**; tank and rope work by **weight × rise of the center of mass**; leaky buckets **in time**.

In [3]:
Z = lambda s: 0.0*s
print("== 2026-10-14: washers about other axes (MATH1342 notes) ==")
check('V-p11', 49*pi/30, wash_sym(sqrt(x) + 2, x**2 + 2, 0, 1), vol_slices_y(np.sqrt, lambda s: s*s, 0, 1, 0, 1, axis_y=-2), ntol=1e-8, note='axis y = -2; numeric: shells in y')
check('V-p12', 198*pi/5, wash_sym(7 - (x**2 + 1), 7 - (3 - x), -2, 1), vol_slices_y(lambda s: 3 - s, lambda s: s*s + 1, -2, 1, 1, 5, axis_y=7, brk=[2]), ntol=1e-8, note='axis y = 7')
check('V-p13', 48*pi/5, wash_sym(sqrt(8*x), x**2, 0, 2), vol_slices_y(lambda s: np.sqrt(8*s) + 1, lambda s: s*s + 1, 0, 2, 1, 5, axis_y=1), ntol=1e-8, note='axis y = 1')
print("== 2026-10-14: cylindrical shells ==")
check('S-guided-b', 1, sp.Max(*sp.solve(x - x**2, x)))
check('S-guided-val', pi/6, shell_sym(x, x - x**2, 0, 1), vol_slices_y(lambda s: s - s*s, Z, 0, 1, 0, 0.25, axis_x=0), ntol=1e-8, note='MATH1342 p. 5; numeric: washers in y')
check('S-indep', pi/6, shell_sym(x, x - x**2, 0, 1), vol_slices_y(lambda s: s, lambda s: s*s, 0, 1, 0, 1, axis_x=0), ntol=1e-8, note='between y = x and y = x^2')
check('S-p1', 8*pi/3, shell_sym(x - 1, 4*x - x**2 - 3, 1, 3), vol_slices_y(lambda s: 4*s - s*s, lambda s: 3 + 0*s, 1, 3, 3, 4, axis_x=1), ntol=1e-8, note='MATH1342 p. 7, axis x = 1')
check('S-p2', 16*pi/3, shell_sym(4 - x, 4*x - x**2 - 3, 1, 3), vol_slices_y(lambda s: 4*s - s*s, lambda s: 3 + 0*s, 1, 3, 3, 4, axis_x=4), ntol=1e-8, note='axis x = 4')
check('S-p3-a', -2, sp.Min(*sp.solve(x**2 + 1 - (3 - x), x))); check('S-p3-b', 1, sp.Max(*sp.solve(x**2 + 1 - (3 - x), x)))
check('S-p3', 99*pi/2, shell_sym(5 - x, (3 - x) - (x**2 + 1), -2, 1), vol_slices_y(lambda s: 3 - s, lambda s: s*s + 1, -2, 1, 1, 5, axis_x=5, brk=[2]), ntol=1e-8, note='MATH1342 p. 8, axis x = 5')
check('S-p4', 232*pi/15, shell_sym(x + 2, sqrt(8*x) - x**2, 0, 2), vol_slices_y(lambda s: np.sqrt(8*s) + 1, lambda s: s*s + 1, 0, 2, 1, 5, axis_x=-2), ntol=1e-8, note='MATH1342 p. 9, axis x = -2')
check('S-p5', 2*pi**2, shell_sym(x, sin(x), 0, pi), vol_slices_y(np.sin, Z, 0, np.pi, 0, 1, axis_x=0), ntol=1e-8, note='sin x about the y-axis')
check('S-p6', pi/9, shell_sym(x, 1 - x**R(1, 4), 0, 1), vol_slices_y(lambda s: 1 + 0*s, lambda s: s**0.25, 0, 1, 0, 1, axis_x=0), ntol=1e-8)
check('S-p7', 2*pi, shell_sym(x, sin(x**2), 0, sqrt(pi)), vol_slices_y(lambda s: np.sin(s*s), Z, 0, np.sqrt(np.pi), 0, 1, axis_x=0), ntol=1e-8)
check('S-p8-a', R(1, 2), sp.Min(*sp.solve(-x**2 + 7*x - (x**2 + 3), x))); check('S-p8-b', 3, sp.Max(*sp.solve(-x**2 + 7*x - (x**2 + 3), x)))
check('S-p8', 875*pi/48, shell_sym(x, (-x**2 + 7*x) - (x**2 + 3), R(1, 2), 3), vol_slices_y(lambda s: -s*s + 7*s, lambda s: s*s + 3, 0.5, 3, 3.25, 12, axis_x=0), ntol=1e-8)
check('S-p9', 8*pi, shell_sym(5 - x, 4*x - x**2 - 3, 1, 3), vol_slices_y(lambda s: 4*s - s*s, lambda s: 3 + 0*s, 1, 3, 3, 4, axis_x=5), ntol=1e-8, note='axis x = 5')
check('S-p10', 8*pi, shell_sym(4 - x, (3*x - x**2) - x, 0, 2), vol_slices_y(lambda s: 3*s - s*s, lambda s: s, 0, 2, 0, 2.25, axis_x=4, brk=[2]), ntol=1e-8, note='axis x = 4')
check('S-m3', pi, I(2*pi*x*exp(-x**2), (x, 0, sp.oo)), mq(lambda s: mp.pi*(-mp.log(s)), [0, 1]), note='improper; numeric: washers in y, x = sqrt(-ln y)')
check('S-m4', 6*pi**2, I(4*pi*(u + 3)*sqrt(1 - u**2), (u, -1, 1)),
      vol_slices_y(lambda s: np.sqrt(np.maximum(0, 1 - (s - 3)**2)), lambda s: -np.sqrt(np.maximum(0, 1 - (s - 3)**2)), 2, 4, -1, 1, axis_x=0), ntol=1e-8, note='torus; shells with u = x - 3')
print("== 2026-10-14: Mixed 27-29 ==")
check('M-S1', 63*pi/2, shell_sym(4 - x, (x + 2) - x**2, -1, 2), vol_slices_y(lambda s: s + 2, lambda s: s*s, -1, 2, 0, 4, axis_x=4, brk=[1]), ntol=1e-8, note='axis x = 4')
check('M-V4', 117*pi/5, wash_sym((x + 2) + 1, x**2 + 1, -1, 2), vol_slices_y(lambda s: s + 2, lambda s: s*s, -1, 2, 0, 4, axis_y=-1, brk=[1]), ntol=1e-8, note='axis y = -1; numeric: shells in y')
check('M-S3', pi*(E - 1), shell_sym(x, exp(x**2), 0, 1), vol_slices_y(lambda s: np.exp(s*s), Z, 0, 1, 0, np.e, axis_x=0, brk=[1]), ntol=1e-8)
print("== 2026-10-19: final items on shells ==")
check('F-S2', 16*pi/3, shell_sym(x + 1, 2*x - x**2, 0, 2), vol_slices_y(lambda s: 2*s - s*s, Z, 0, 2, 0, 1, axis_x=-1), ntol=1e-8, note='axis x = -1')
check('F-S3', 36*pi, shell_sym(x, 2*sqrt(9 - x**2), 0, 3),
      vol_slices_y(lambda s: np.sqrt(np.maximum(0, 9 - s*s)), lambda s: -np.sqrt(np.maximum(0, 9 - s*s)), 0, 3, -3, 3, axis_x=0), ntol=1e-8, note='= (4/3) pi 3^3')

== 2026-10-14: washers about other axes (MATH1342 notes) ==


OK       V-p11                  claim=49*pi/30                       value=5.13126800086333       numeric=5.131268000863     axis y = -2; numeric: shells in y
OK       V-p12                  claim=198*pi/5                       value=124.407069082156       numeric=124.4070690822     axis y = 7


OK       V-p13                  claim=48*pi/5                        value=30.159289474462        numeric=30.15928947446     axis y = 1
== 2026-10-14: cylindrical shells ==
OK       S-guided-b             claim=1                              value=1                      numeric=                   
OK       S-guided-val           claim=pi/6                           value=0.523598775598299      numeric=0.5235987755983    MATH1342 p. 5; numeric: washers in y
OK       S-indep                claim=pi/6                           value=0.523598775598299      numeric=0.5235987755983    between y = x and y = x^2
OK       S-p1                   claim=8*pi/3                         value=8.37758040957278       numeric=8.377580409573     MATH1342 p. 7, axis x = 1
OK       S-p2                   claim=16*pi/3                        value=16.7551608191456       numeric=16.75516081915     axis x = 4
OK       S-p3-a                 claim=-2                             value=-2                     num

OK       S-p4                   claim=232*pi/15                      value=48.5899663755221       numeric=48.58996637552     MATH1342 p. 9, axis x = -2
OK       S-p5                   claim=2*pi**2                        value=19.7392088021787       numeric=19.73920880218     sin x about the y-axis


OK       S-p6                   claim=pi/9                           value=0.349065850398866      numeric=0.3490658503989    
OK       S-p7                   claim=2*pi                           value=6.28318530717959       numeric=6.28318530718      
OK       S-p8-a                 claim=1/2                            value=0.5                    numeric=                   
OK       S-p8-b                 claim=3                              value=3                      numeric=                   
OK       S-p8                   claim=875*pi/48                      value=57.2686160810639       numeric=57.26861608106     
OK       S-p9                   claim=8*pi                           value=25.1327412287183       numeric=25.13274122872     axis x = 5
OK       S-p10                  claim=8*pi                           value=25.1327412287183       numeric=25.13274122872     axis x = 4
OK       S-m3                   claim=pi                             value=3.14159265358979       

OK       S-m4                   claim=6*pi**2                        value=59.2176264065362       numeric=59.21762640654     torus; shells with u = x - 3
== 2026-10-14: Mixed 27-29 ==
OK       M-S1                   claim=63*pi/2                        value=98.9601685880785       numeric=98.96016858808     axis x = 4
OK       M-V4                   claim=117*pi/5                       value=73.5132680940012       numeric=73.513268094       axis y = -1; numeric: shells in y
OK       M-S3                   claim=pi*(-1 + E)                    value=5.39814156908377       numeric=5.398141569084     
== 2026-10-19: final items on shells ==
OK       F-S2                   claim=16*pi/3                        value=16.7551608191456       numeric=16.75516081915     axis x = -1
OK       F-S3                   claim=36*pi                          value=113.097335529233       numeric=113.0973355292     = (4/3) pi 3^3


True

In [4]:
print("== §3.7 mass and density (K) ==")
check('K-guided-b', 2, sp.Max(*sp.solve(sp.Eq(x - x**2, -x), x)))
check('K-guided-val', R(4, 3), plate_sym(x, x - x**2, -x, 0, 2)[0], plate_num(lambda s: s, lambda s: s - s*s, lambda s: -s, 0, 2)[0], ntol=1e-8, note='notes p. 5; numeric: double integral')
check('K-indep', R(45, 2), plate_sym(x**2 + 1, 0*x, 2*x - 6, 0, 3)[0], plate_num(lambda s: s*s + 1, lambda s: 0.0, lambda s: 2*s - 6, 0, 3)[0], ntol=1e-8, note='triangle below the x-axis')
check('K-p1-M', 8, I(3*x**2, (x, 0, 2)), mq(lambda s: 3*s**2, [0, 2]))
check('K-p1-xm', sp.cbrt(4), mid_sym(3*x**2, 0, 2), mid_num(lambda s: 3*s**2, 0, 2))
check('K-p2-a', -1, sp.Min(*sp.solve(sp.Eq(x**2, 2 - x**2), x))); check('K-p2-b', 1, sp.Max(*sp.solve(sp.Eq(x**2, 2 - x**2), x)))
check('K-p3-L', 1 - exp(-2), I(exp(-x), (x, 0, 2)), mq(lambda s: mp.e**(-s), [0, 2]), note='textbook Ex. 3.7.6')
check('K-p3-R', exp(-2) - exp(-4), I(exp(-x), (x, 2, 4)), mq(lambda s: mp.e**(-s), [2, 4]))
check('K-p3-xm', log(2/(1 + exp(-4))), mid_sym(exp(-x), 0, 4), mid_num(lambda s: mp.e**(-s), 0, 4))
check('K-p4-M', 36, I(x + 7, (x, 0, 4)), mq(lambda s: s + 7, [0, 4]))
check('K-p4-xm', sqrt(85) - 7, mid_sym(x + 7, 0, 4), mid_num(lambda s: s + 7, 0, 4))
check('K-p5-M', 1, I(sin(x), (x, 0, pi/2)), mq(mp.sin, [0, mp.pi/2]))
check('K-p5-xm', pi/3, mid_sym(sin(x), 0, pi/2), mid_num(mp.sin, 0, mp.pi/2))
c6 = sp.solve(sp.Eq(I(c*x**2*(2 - 2*x/5), (x, 0, 5)), 25), c)[0]
check('K-p6-c', R(6, 5), c6, 25/plate_num(lambda s: s*s, lambda s: 2 - 0.4*s, lambda s: 0.0, 0, 5)[0], ntol=1e-8, note='triangle (0,0), (0,2), (5,0)')
check('K-p7', R(2, 3)*log(R(5, 2)), I(4/(9 - x**2), (x, 1, 2)), mq(lambda s: 4/(9 - s*s), [1, 2]), note='partial fractions')
check('K-p8', R(153, 2)*log(3) - 18, I(x*log(x), (x, 3, 9)), mq(lambda s: s*mp.log(s), [3, 9]), note='by parts')
check('K-m3', log((1 + exp(10))/2), mid_sym(c*exp(x), 0, 10), mid_num(lambda s: 7*mp.e**s, 0, 10), note='c cancels (numeric run with c = 7)')
check('K-m4-M', 1, I(1/(1 + x)**2, (x, 0, sp.oo)), mq(lambda s: 1/(1 + s)**2, [0, mp.inf]), note='improper')
check('K-m4-xm', 1, pick_root(sp.Eq(I(1/(1 + x)**2, (x, 0, t)), R(1, 2)), t, 0, 100), mp.findroot(lambda s: mq(lambda z: 1/(1 + z)**2, [0, s]) - 0.5, 2))
print("== §3.8 center of mass (G) ==")
gm = wire_sym(4*x**2 + 2, 0, 3)
check('G-guided-M', 42, gm[0], mq(lambda s: 4*s*s + 2, [0, 3])); check('G-guided-Mx', 90, I(x*(4*x**2 + 2), (x, 0, 3)), mq(lambda s: s*(4*s*s + 2), [0, 3]))
check('G-guided-xbar', R(15, 7), gm[1], mq(lambda s: s*(4*s*s + 2), [0, 3])/42)
gi = plate_sym(2*x + 1, x, x**2, 0, 1); ni = plate_num(lambda s: 2*s + 1, lambda s: s, lambda s: s*s, 0, 1)
check('G-indep-M', R(1, 3), gi[0], ni[0], ntol=1e-8); check('G-indep-x', R(11, 20), gi[1], ni[1], ntol=1e-8); check('G-indep-y', R(9, 20), gi[2], ni[2], ntol=1e-8, note='numeric: double integral of y')
check('G-p1-M', R(2, 3)*(E**12 - E**9), I(d9, (x, 3, 4)), mq(n9, [3, 4]))
check('G-p1-Mx', R(2, 9)*(11*E**12 - 8*E**9), I(x*d9, (x, 3, 4)), mq(lambda s: s*n9(s), [3, 4]), note='the corrected notes p. 9 moment')
check('G-p1-x', (11*E**12 - 8*E**9)/(3*(E**12 - E**9)), wire_sym(d9, 3, 4)[1], mq(lambda s: s*n9(s), [3, 4])/mq(n9, [3, 4]))
g2 = plate_sym(x**2, 2/x**2, 0*x, 1, 2)
check('G-p2-x', R(3, 2), g2[1], n12[1], ntol=1e-8); check('G-p2-y', R(1, 2), g2[2], n12[2], ntol=1e-8)
check('G-p3-a', -1, sp.Min(*sp.solve(sp.Eq(x**2, 2*x + 3), x))); check('G-p3-b', 3, sp.Max(*sp.solve(sp.Eq(x**2, 2*x + 3), x)))
check('G-p4', R(13, 5), R(2*(-1) + 5*2 + 3*6, 2 + 5 + 3), np.dot([2, 5, 3], [-1, 2, 6])/10)
check('G-p5', (1 - 5*exp(-4))/(1 - exp(-4)), wire_sym(exp(-x), 0, 4)[1], mq(lambda s: s*mp.e**(-s), [0, 4])/mq(lambda s: mp.e**(-s), [0, 4]), note='textbook Ex. 3.8.5')
check('G-p6-Mx', R(25225, 12), I(x*(3*x**2 + 2*x + 4), (x, 2, 7)), mq(lambda s: s*(3*s*s + 2*s + 4), [2, 7]))
check('G-p6-x', R(1009, 192), wire_sym(3*x**2 + 2*x + 4, 2, 7)[1], mq(lambda s: s*(3*s*s + 2*s + 4), [2, 7])/mq(lambda s: 3*s*s + 2*s + 4, [2, 7]))
g7 = plate_sym(2 - x**2, x, -x, 0, 1); n7 = plate_num(lambda s: 2 - s*s, lambda s: s, lambda s: -s, 0, 1)
check('G-p7-M', R(3, 2), g7[0], n7[0], ntol=1e-8); check('G-p7-x', R(28, 45), g7[1], n7[1], ntol=1e-8); check('G-p7-y', 0, g7[2], n7[2], ntol=1e-8, note='symmetry')
g8 = plate_sym(x, x**2, 1 + 0*x, 1, 2); n8 = plate_num(lambda s: s, lambda s: s*s, lambda s: 1.0, 1, 2)
check('G-p8-x', R(232, 135), g8[1], n8[1], ntol=1e-8, note='textbook Ex. 3.8.7'); check('G-p8-y', 2, g8[2], n8[2], ntol=1e-8)
g9 = plate_sym(1, sqrt(x), x, 0, 1); n9b = plate_num(lambda s: 1.0, np.sqrt, lambda s: s, 0, 1)
check('G-p9-x', R(2, 5), g9[1], n9b[1], ntol=1e-8, note='centroid'); check('G-p9-y', R(1, 2), g9[2], n9b[2], ntol=1e-8)
PM = [(3, -1, 0), (6, 0, 1), (4, 1, 0), (9, 0, -1)]
check('G-m3-x', R(1, 22), sum(R(m)*a for m, a, b in PM)/sum(m for m, a, b in PM), np.dot([m for m, a, b in PM], [a for m, a, b in PM])/22)
check('G-m3-y', -R(3, 22), sum(R(m)*b for m, a, b in PM)/sum(m for m, a, b in PM), np.dot([m for m, a, b in PM], [b for m, a, b in PM])/22)
gh = plate_sym(1, sqrt(16 - x**2), 0*x, -4, 4); nh = plate_num(lambda s: 1.0, lambda s: np.sqrt(max(0.0, 16 - s*s)), lambda s: 0.0, -4, 4)
check('G-m4-x', 0, gh[1], nh[1], ntol=1e-8); check('G-m4-y', 16/(3*pi), gh[2], nh[2], ntol=1e-8, note='half-disk centroid')

== §3.7 mass and density (K) ==
OK       K-guided-b             claim=2                              value=2                      numeric=                   
OK       K-guided-val           claim=4/3                            value=1.33333333333333       numeric=1.333333333333     notes p. 5; numeric: double integral
OK       K-indep                claim=45/2                           value=22.5                   numeric=22.5               triangle below the x-axis
OK       K-p1-M                 claim=8                              value=8                      numeric=8                  
OK       K-p1-xm                claim=2**(2/3)                       value=1.5874010519682        numeric=1.587401051968     
OK       K-p2-a                 claim=-1                             value=-1                     numeric=                   
OK       K-p2-b                 claim=1                              value=1                      numeric=                   
OK       K-p3-L          

OK       K-p3-xm                claim=log(2/(exp(-4) + 1))           value=0.674997252642136      numeric=0.6749972526421    
OK       K-p4-M                 claim=36                             value=36                     numeric=36                 
OK       K-p4-xm                claim=-7 + sqrt(85)                  value=2.21954445729289       numeric=2.219544457293     
OK       K-p5-M                 claim=1                              value=1                      numeric=1                  
OK       K-p5-xm                claim=pi/3                           value=1.0471975511966        numeric=1.047197551197     
OK       K-p6-c                 claim=6/5                            value=1.2                    numeric=1.2                triangle (0,0), (0,2), (5,0)


OK       K-p7                   claim=2*log(5/2)/3                   value=0.610860487916103      numeric=0.6108604879161    partial fractions
OK       K-p8                   claim=-18 + 153*log(3)/2             value=66.0438400831104       numeric=66.04384008311     by parts
OK       K-m3                   claim=log(1/2 + exp(10)/2)           value=9.30689821833927       numeric=9.306898218339     c cancels (numeric run with c = 7)
OK       K-m4-M                 claim=1                              value=1                      numeric=1                  improper
OK       K-m4-xm                claim=1                              value=1                      numeric=1                  
== §3.8 center of mass (G) ==
OK       G-guided-M             claim=42                             value=42                     numeric=42                 


OK       G-guided-Mx            claim=90                             value=90                     numeric=90                 
OK       G-guided-xbar          claim=15/7                           value=2.14285714285714       numeric=2.142857142857     
OK       G-indep-M              claim=1/3                            value=0.333333333333333      numeric=0.3333333333333    
OK       G-indep-x              claim=11/20                          value=0.55                   numeric=0.55               
OK       G-indep-y              claim=9/20                           value=0.45                   numeric=0.45               numeric: double integral of y
OK       G-p1-M                 claim=-2*exp(9)/3 + 2*exp(12)/3      value=103101.138327619       numeric=103101.1383276     
OK       G-p1-Mx                claim=-16*exp(9)/9 + 22*exp(12)/9    value=383439.563152987       numeric=383439.563153      the corrected notes p. 9 moment
OK       G-p1-x                 claim=(-8*exp(9) + 11*exp(

OK       G-p6-x                 claim=1009/192                       value=5.25520833333333       numeric=5.255208333333     
OK       G-p7-M                 claim=3/2                            value=1.5                    numeric=1.5                
OK       G-p7-x                 claim=28/45                          value=0.622222222222222      numeric=0.6222222222222    
OK       G-p7-y                 claim=0                              value=0                      numeric=0                  symmetry
OK       G-p8-x                 claim=232/135                        value=1.71851851851852       numeric=1.718518518519     textbook Ex. 3.8.7
OK       G-p8-y                 claim=2                              value=2                      numeric=2                  
OK       G-p9-x                 claim=2/5                            value=0.4                    numeric=0.4                centroid
OK       G-p9-y                 claim=1/2                            value=0.5      

True

In [5]:
print("== §3.9 work (W) ==")
W98 = mp.mpf('9.8')
check('W-guided-val', 2940000*pi, I(1000*pi*(5*y/12)**2*g98*(12 - y), (y, 0, 12)), tank_cm(lambda s: 1000*W98*mp.pi*(5*s/12)**2, 12, 12), note='notes p. 18; numeric: weight x rise of cm')
check('W-indep', 2496000, I(R(624, 10)*20*8*(30 - y), (y, 0, 10)), tank_cm(lambda s: mp.mpf('62.4')*160, 10, 30), note='textbook Suggested Ex. 7')
check('W-p1', 2500, I(2*(50 - y), (y, 0, 50)), tank_cm(lambda s: 2, 50, 50), note='notes p. 16')
check('W-p2-b', 9, R(3, 4)*12)
check('W-p2', 20947500, I(1000*5**2*g98*(12 + 2 - y), (y, 0, 9)), tank_cm(lambda s: 1000*25*W98, 9, 14), note='notes p. 17')
check('W-p3', 74880000*pi, I(75*pi*((100 - y)/5)**2*(100 - y), (y, 0, 80)), tank_cm(lambda s: 75*mp.pi*((100 - s)/5)**2, 80, 100), note='notes p. 19')
check('W-p4-k', 30, k21, 10/(4/12)); check('W-p4-W', R(15, 4), I(k21*x, (x, 0, R(1, 2))), mq(lambda s: 30*s, [0, 0.5]))
check('W-p5', 20, I(10*x, (x, 0, -2)), mq(lambda s: 10*s, [0, -2]))
check('W-p6', 30, sp.solve(sp.Eq(I(k*x, (x, 0, 4)), 240), k)[0], 240/mq(lambda s: s, [0, 4]))
check('W-p7', 2700, I(32 - R(2, 10)*(y/2), (y, 0, 100)), mq(lambda s: (32 - 0.2*s)*2, [0, 50]), note='numeric: in time (50 s at 2 ft/s)')
check('W-p8', 180, R(1, 2)*10*vt.subs(t, 3)**2 - R(1, 2)*10*vt.subs(t, 0)**2, mq(lambda s: 10*(-2)*(-2*s), [0, 3]), note="numeric: integral of m x'' x' dt")
check('W-p9', 4575, I(80 - R(1, 2)*(y/4), (y, 0, 60)), mq(lambda s: (80 - 0.5*s)*4, [0, 15]), note='numeric: in time (15 s at 4 ft/s)')
check('W-p10-k', 130, sp.solve(sp.Eq(k*R(3, 10), 39), k)[0], 39/0.3)
check('W-p10-W', R(65, 4), I(130*x, (x, 0, -R(1, 2))), mq(lambda s: 130*s, [0, -0.5]))
check('W-m3', 2521, I(50 - y/2, (y, 0, 98)) + I(1, (y, 98, 120)), mq(lambda s: max(50 - s, 1)*2, [0, 49, 60]), note='a force in two pieces; numeric in time')
check('W-m4', 8 + log(5), I((x**2 + x + 1)/(x + 1), (x, 0, 4)), mq(lambda s: (s*s + s + 1)/(s + 1), [0, 4]), note='long division')
print("== Mixed 30-35 ==")
check('M-K1', R(14, 3), I(sqrt(x), (x, 1, 4)), mq(mp.sqrt, [1, 4]))
check('M-G1', R(93, 35), wire_sym(sqrt(x), 1, 4)[1], mq(lambda s: s*mp.sqrt(s), [1, 4])/mq(mp.sqrt, [1, 4]))
check('M-W1', 490000*pi, I(1000*g98*pi*2**2*(5 - y), (y, 0, 5)), tank_cm(lambda s: 1000*W98*4*mp.pi, 5, 5))
check('M-K2', R(3, 20), plate_sym(x, sqrt(x), x**2, 0, 1)[0], plate_num(lambda s: s, np.sqrt, lambda s: s*s, 0, 1)[0], ntol=1e-8)
kk = sp.solve(sp.Eq(I(k*x, (x, 0, R(1, 5))), 6), k)[0]
check('M-W2', 18, I(kk*x, (x, R(1, 5), R(2, 5))), mq(lambda s: 300*s, [0.2, 0.4]))
gq = plate_sym(x, 2 + 0*x, 0*x, 0, 2); nq = plate_num(lambda s: s, lambda s: 2.0, lambda s: 0.0, 0, 2)
check('M-G2-x', R(4, 3), gq[1], nq[1], ntol=1e-8); check('M-G2-y', 1, gq[2], nq[2], ntol=1e-8)
print("== Final check, items 35-42 ==")
check('F-K2', R(28, 3), plate_sym(1 + x, 4 - x**2, 0*x, 0, 2)[0], plate_num(lambda s: 1 + s, lambda s: 4 - s*s, lambda s: 0.0, 0, 2)[0], ntol=1e-8)
check('F-K3', 3, mid_sym(1/x, 1, 9), mid_num(lambda s: 1/s, 1, 9), note='the geometric mean of 1 and 9')
check('F-G2', R(7, 6), wire_sym(1 + x, 0, 2)[1], mq(lambda s: s*(1 + s), [0, 2])/mq(lambda s: 1 + s, [0, 2]))
f3 = plate_sym(1, 9 + 0*x, x**2, -3, 3); nf3 = plate_num(lambda s: 1.0, lambda s: 9.0, lambda s: s*s, -3, 3)
check('F-G3-x', 0, f3[1], nf3[1], ntol=1e-8); check('F-G3-y', R(27, 5), f3[2], nf3[2], ntol=1e-8, note='textbook Additional Ex. 10')
kf = sp.solve(sp.Eq(k*R(3, 10), 12), k)[0]
check('F-W2', 5, I(kf*x, (x, 0, R(1, 2))), mq(lambda s: 40*s, [0, 0.5]))
check('F-W3', 5880, I(2*g98*(10 - y), (y, 0, 10)) + 50*g98*10, tank_cm(lambda s: 2*W98, 10, 10) + 50*W98*10, note='cable sliced + bucket as a whole')
print(f"\n{len(ROWS)} rows, mismatches: {[r_['id'] for r_ in ROWS if not r_['ok']]}")

== §3.9 work (W) ==
OK       W-guided-val           claim=2940000*pi                     value=9236282.40155399       numeric=9236282.401554     notes p. 18; numeric: weight x rise of cm
OK       W-indep                claim=2496000                        value=2496000                numeric=2496000            textbook Suggested Ex. 7
OK       W-p1                   claim=2500                           value=2500                   numeric=2500               notes p. 16
OK       W-p2-b                 claim=9                              value=9                      numeric=                   
OK       W-p2                   claim=20947500                       value=20947500               numeric=20947500           notes p. 17
OK       W-p3                   claim=74880000*pi                    value=235242457.900804       numeric=235242457.9008     notes p. 19
OK       W-p4-k                 claim=30                             value=30                     numeric=30                 


OK       M-K2                   claim=3/20                           value=0.15                   numeric=0.15               
OK       M-W2                   claim=18                             value=18                     numeric=18                 
OK       M-G2-x                 claim=4/3                            value=1.33333333333333       numeric=1.333333333333     
OK       M-G2-y                 claim=1                              value=1                      numeric=1                  
== Final check, items 35-42 ==
OK       F-K2                   claim=28/3                           value=9.33333333333333       numeric=9.333333333333     
OK       F-K3                   claim=3                              value=3                      numeric=3                  the geometric mean of 1 and 9
OK       F-G2                   claim=7/6                            value=1.16666666666667       numeric=1.166666666667     
OK       F-G3-x                 claim=0                   

## 3. Typed-expression answers, checked with SymPy and with the guide's own grader

Each intended expression is written from the problem statement, not copied from the key. `js_eval.js` (expression keys) and `js_eval_num.js` (numeric keys, new on 2026-10-20) extract the parser and grader from `app.js` and run them on every key in the guide, old and new.

In [6]:
TR = standard_transformations + (implicit_multiplication_application, convert_xor)
X, Y = sp.Symbol('x', positive=True), sp.Symbol('y', positive=True)
js2sp = lambda s_: parse_expr(s_, local_dict={'e': sp.E, 'pi': sp.pi, 'ln': sp.log, 'sqrt': sp.sqrt, 'x': X, 'y': Y}, transformations=TR)
KEYS = {k_['vid']: k_ for k_ in json.load(open(BUILD / 'answer_keys_full.json', encoding='utf-8'))}
BEFORE14 = json.load(open(VER / 'guide_state_before_2026-10-14.json', encoding='utf-8'))
BEFORE19 = json.load(open(VER / 'guide_state_before_2026-10-19.json', encoding='utf-8'))
b14 = {k_['vid']: k_ for k_ in BEFORE14['answer_keys']}; b19 = {k_['vid']: k_ for k_ in BEFORE19['answer_keys']}
NEW = [v for v in KEYS if v not in b14]                     # everything added on 2026-10-14 or 2026-10-19
EXPR_REF = {                                                # written from each problem statement
 'V-p11-R': sqrt(X) - (-2), 'V-p11-r': X**2 - (-2), 'V-p12-R': 7 - (X**2 + 1), 'V-p12-r': 7 - (-X + 3),
 'V-p13-R': (sqrt(8*X) + 1) - 1, 'V-p13-r': (X**2 + 1) - 1,
 'S-guided-r': X - 0, 'S-guided-h': (X - X**2) - 0, 'S-guided-shell': 2*pi*(X - 0)*(X - X**2),
 'S-p1-r': X - 1, 'S-p1-h': (4*X - X**2) - 3, 'S-p2-r': 4 - X, 'S-p3-h': (-X + 3) - (X**2 + 1),
 'S-p4-r': X - (-2), 'S-p4-h': (sqrt(8*X) + 1) - (X**2 + 1), 'S-p10-r': 4 - X, 'S-p10-h': (3*X - X**2) - X,
 'K-guided-h': (X - X**2) - (-X), 'K-guided-dm': X*((X - X**2) - (-X)), 'K-p2-dm': 3**X*((2 - X**2) - X**2),
 'K-p6-h': 2 + (0 - 2)/sp.Integer(5 - 0)*X,                 # the line through (0, 2) and (5, 0)
 'G-guided-mom': X*(4*X**2 + 2), 'G-p3-dm': 2**X*((2*X + 3) - X**2), 'G-p3-yf': ((2*X + 3) + X**2)/2,
 'W-guided-r': 5*Y/12, 'W-guided-F': 1000*g98*pi*(5*Y/12)**2, 'W-guided-d': 12 - Y, 'W-p2-d': (12 + 2) - Y, 'W-p3-r': 20*(100 - Y)/100,
}
new_expr = sorted(v for v in NEW if KEYS[v]['kind'] != 'num')
assert sorted(EXPR_REF) == new_expr, set(new_expr) ^ set(EXPR_REF)
EXPR_OK = {}
for vid, ref in EXPR_REF.items():
    k_ = KEYS[vid]
    ok = sp.simplify(js2sp(k_['answer']) - ref) == 0
    wrong_differ = all(sp.simplify(js2sp(w_['f']) - ref) != 0 for w_ in k_['wrong'] if 'f' in w_)
    EXPR_OK[vid] = ok and wrong_differ
    print(f"{'OK      ' if EXPR_OK[vid] else 'MISMATCH'} {vid:<15} key = {k_['answer']:<18} intended = {str(ref):<26} every listed wrong expression differs: {wrong_differ}")
JSE = subprocess.run(['node', str(BUILD / 'js_eval.js')], capture_output=True, text=True, cwd=BUILD); print("\njs_eval.js:    ", JSE.stdout.strip())
JSN = subprocess.run(['node', str(BUILD / 'js_eval_num.js')], capture_output=True, text=True, cwd=BUILD); print("js_eval_num.js:", JSN.stdout.strip())
jse = {r_['vid']: r_ for r_ in json.load(open(BUILD / 'js_eval.json', encoding='utf-8'))}
jsn = {r_['vid']: r_ for r_ in json.load(open(BUILD / 'js_eval_num.json', encoding='utf-8'))}
js_ok = 'PROBLEMS' not in JSE.stdout and 'PROBLEMS' not in JSN.stdout
# the grader's own evaluation of each new expression key, at its sample points, must equal the SymPy intended expression there
worst_js = 0.0
for vid, ref in EXPR_REF.items():
    r_ = jse[vid]; v_ = X if KEYS[vid]['var'] == 'x' else Y
    worst_js = max(worst_js, max(abs(a_ - float(ref.subs(v_, p_))) for a_, p_ in zip(r_['answer'], r_['pts'])))
    js_ok &= r_['self_ok'] and r_['withC_ok'] and all(not w_['ok'] and w_['targeted'] for w_ in r_['wrong'])
for vid in NEW:
    if KEYS[vid]['kind'] == 'num':
        r_ = jsn[vid]; js_ok &= r_['self_ok'] and all(not w_['ok'] and w_['targeted'] for w_ in r_['wrong']) and r_.get('units_ok', True) and r_.get('wrong_with_unit', True)
print(f"grader values at the sample points vs SymPy, worst difference: {worst_js:.1e} | every new key graded correctly: {js_ok}")
UNITS = {v: KEYS[v]['units'] for v in NEW if KEYS[v].get('units')}
print(f"inputs that take units ({len(UNITS)}):", UNITS)

OK       V-p11-R         key = sqrt(x)+2          intended = sqrt(x) + 2                every listed wrong expression differs: True
OK       V-p11-r         key = x^2+2              intended = x**2 + 2                   every listed wrong expression differs: True
OK       V-p12-R         key = 6-x^2              intended = 6 - x**2                   every listed wrong expression differs: True
OK       V-p12-r         key = 4+x                intended = x + 4                      every listed wrong expression differs: True
OK       V-p13-R         key = sqrt(8*x)          intended = 2*sqrt(2)*sqrt(x)          every listed wrong expression differs: True
OK       V-p13-r         key = x^2                intended = x**2                       every listed wrong expression differs: True
OK       S-guided-r      key = x                  intended = x                          every listed wrong expression differs: True
OK       S-guided-h      key = x-x^2              intended = -x**2 + x      

OK       K-p2-dm         key = 3^x*(2-2*x^2)      intended = 3**x*(2 - 2*x**2)          every listed wrong expression differs: True
OK       K-p6-h          key = 2-2*x/5            intended = 2 - 2*x/5                  every listed wrong expression differs: True
OK       G-guided-mom    key = x*(4*x^2+2)        intended = x*(4*x**2 + 2)             every listed wrong expression differs: True
OK       G-p3-dm         key = 2^x*(2*x+3-x^2)    intended = 2**x*(-x**2 + 2*x + 3)     every listed wrong expression differs: True
OK       G-p3-yf         key = (x^2+2*x+3)/2      intended = x**2/2 + x + 3/2           every listed wrong expression differs: True
OK       W-guided-r      key = 5*y/12             intended = 5*y/12                     every listed wrong expression differs: True
OK       W-guided-F      key = 9800*pi*(5*y/12)^2 intended = 30625*pi*y**2/18           every listed wrong expression differs: True
OK       W-guided-d      key = 12-y               intended = 12 - y         


js_eval.js:     expression keys: 84 | reference accepted: 84 | with +C accepted: 84 | wrong answers rejected with their targeted message: 76/76
js_eval_num.js: numeric keys: 270 | reference accepted: 270 | wrong values rejected with their own message: 206/206 | unit handling correct: 30/30
grader values at the sample points vs SymPy, worst difference: 1.2e-10 | every new key graded correctly: True
inputs that take units (30): {'K-indep': 'kg', 'K-p3-L': 'slugs', 'K-p3-R': 'slugs', 'K-p3-xm': 'ft', 'K-m4-M': 'kg', 'K-m4-xm': 'm', 'G-p4': 'm', 'G-p5': 'ft', 'W-guided-val': 'J', 'W-indep': 'ft-lb', 'W-p1': 'ft-lb', 'W-p2': 'J', 'W-p3': 'ft-lb', 'W-p4-k': 'lb/ft', 'W-p4-W': 'ft-lb', 'W-p5': 'J', 'W-p6': 'N/m', 'W-p7': 'ft-lb', 'W-p8': 'J', 'W-p9': 'ft-lb', 'W-p10-k': 'N/m', 'W-p10-W': 'J', 'W-m3': 'ft-lb', 'W-m4': 'J', 'M-K1': 'kg', 'M-G1': 'm', 'M-W1': 'J', 'M-W2': 'J', 'F-W2': 'J', 'F-W3': 'J'}


## 4. Targeted wrong-answer values: each one really is what the described mistake produces

Each value is recomputed in SymPy from the mistake its message describes (the label after `|`, which is also the Wolfram row label). Then every listed wrong value in every new key must be matched by exactly one entry, and must lie outside the grader's tolerance for the right answer.

In [7]:
sh, wa = shell_sym, wash_sym
MIST = {   # 'vid|mistake' : SymPy value of the mistaken computation
 'K-guided-val|top only': I(x*(x - x**2), (x, 0, 2)), 'K-indep|area': I(6 - 2*x, (x, 0, 3)), 'K-indep|swapped': I((x**2 + 1)*(2*x - 6), (x, 0, 3)),
 'K-indep|wire': I(x**2 + 1, (x, 0, 3)), 'K-p1-xm|solve for M': pick_root(sp.Eq(t**3, 8), t, 0, 3), 'K-p1-xm|xbar': I(x*3*x**2, (x, 0, 2))/8,
 'K-p1-xm|middle': R(0 + 2, 2), 'K-p3-xm|middle': R(0 + 4, 2), 'K-p4-xm|middle': R(0 + 4, 2), 'K-p5-xm|middle': pi/4, 'K-m3|middle': 5,
 'K-p6-c|wire': sp.solve(sp.Eq(I(c*x**2, (x, 0, 5)), 25), c)[0], 'K-p7|sign slip': R(2, 3)*((log(1) + log(5)) - (log(2) + log(4))),
 'G-guided-xbar|upside down': R(42, 90), 'G-guided-xbar|middle': R(3, 2), 'G-guided-xbar|moment itself': I(x*(4*x**2 + 2), (x, 0, 3)),
 'G-indep-M|area': I(x - x**2, (x, 0, 1)), 'G-indep-x|centroid': I(x*(x - x**2), (x, 0, 1))/I(x - x**2, (x, 0, 1)),
 'G-indep-y|no half': I((x + x**2)*(2*x + 1)*(x - x**2), (x, 0, 1))/R(1, 3), 'G-indep-y|centroid': I((x**2 - x**4)/2, (x, 0, 1))/I(x - x**2, (x, 0, 1)),
 'G-indep-y|factor x': gi[1],
 'G-p1-Mx|2/3 slip': AS_WRITTEN, 'G-p1-x|slip quotient': AS_WRITTEN/(R(2, 3)*(E**12 - E**9)), 'G-p1-x|mass midpoint': mid_sym(d9, 3, 4), 'G-p1-x|middle': R(7, 2),
 'G-p2-x|centroid': I(x*2/x**2, (x, 1, 2))/I(2/x**2, (x, 1, 2)), 'G-p2-y|top factor': I((2/x**2)*x**2*(2/x**2), (x, 1, 2))/2,
 'G-p4|plain average': R(-1 + 2 + 6, 3), 'G-p5|mass midpoint': mid_sym(exp(-x), 0, 4), 'G-p5|middle': 2, 'G-p6-x|middle': R(9, 2),
 'G-p7-x|centroid': I(x*2*x, (x, 0, 1))/I(2*x, (x, 0, 1)), 'G-p8-y|centroid': I((x**4 - 1)/2, (x, 1, 2))/I(x**2 - 1, (x, 1, 2)),
 'G-m3-y|divide by 4': R(6 - 9, 4), 'G-m4-y|half radius': 2,
 'W-guided-val|no 9.8': I(1000*pi*(5*y/12)**2*(12 - y), (y, 0, 12)), 'W-guided-val|distance y': I(1000*g98*pi*(5*y/12)**2*y, (y, 0, 12)),
 'W-indep|x9.8': I(R(624, 10)*160*g98*(30 - y), (y, 0, 10)), 'W-indep|0..30': I(R(624, 10)*160*(30 - y), (y, 0, 30)),
 'W-indep|distance y': I(R(624, 10)*160*y, (y, 0, 10)), 'W-indep|weight x 30': R(624, 10)*160*10*30,
 'W-p1|helicopter Fd': 2*50*50, 'W-p2-b|tank height': 12,
 'W-p2|no 9.8': I(1000*25*(14 - y), (y, 0, 9)), 'W-p2|0..12': I(1000*25*g98*(14 - y), (y, 0, 12)), 'W-p2|12-y': I(1000*25*g98*(12 - y), (y, 0, 9)),
 'W-p3|x9.8': I(75*g98*pi*((100 - y)/5)**2*(100 - y), (y, 0, 80)), 'W-p3|r=y/5': I(75*pi*(y/5)**2*(100 - y), (y, 0, 80)),
 'W-p4-k|inches': R(10, 4), 'W-p4-W|in-lb': I(R(10, 4)*x, (x, 0, 6)), 'W-p4-W|Fend x d': 30*R(1, 2)*R(1, 2),
 'W-p5|Fend x d': 10*2*2, "W-p5|spring's own force": I(-10*x, (x, 0, -2)), 'W-p6|16k': sp.solve(sp.Eq(16*k, 240), k)[0],
 'W-p7|no leak': 32*100, 'W-p7|dt': I(32 - R(2, 10)*t, (t, 0, 50)), 'W-p7|final weight': (32 - R(2, 10)*50)*100,
 'W-p8|v not squared': R(1, 2)*10*vt.subs(t, 3)*abs(vt.subs(t, 3)),
 'W-p9|no leak': 80*60, 'W-p9|per foot': I(80 - y/2, (y, 0, 60)), 'W-p10-W|39 x 0.5': R(39, 2),
 'W-m3|no stop': I(50 - y/2, (y, 0, 120)), 'W-m4|remainder only': I(1/(x + 1), (x, 0, 4)), 'W-m4|quotient only': I(x, (x, 0, 4)),
 'M-K1|end density': sqrt(4)*3, 'M-G1|mass midpoint': mid_sym(sqrt(x), 1, 4), 'M-G1|middle': R(5, 2),
 'M-W1|weight x 5': 1000*g98*pi*4*5*5, 'M-K2|shell volume': sh(x, sqrt(x) - x**2, 0, 1),
 'M-W2|0..0.4': I(300*x, (x, 0, R(2, 5))), 'M-W2|first 0.2 m again': I(300*x, (x, 0, R(1, 5))), 'M-G2-x|middle': 1,
 'F-S2|r=x': sh(x, 2*x - x**2, 0, 2), 'F-S3|top half': sh(x, sqrt(9 - x**2), 0, 3),
 'F-K2|area': I(4 - x**2, (x, 0, 2)), 'F-K2|wire': I(1 + x, (x, 0, 2)), 'F-K3|middle': 5, 'F-G2|moment': I(x*(1 + x), (x, 0, 2)), 'F-G2|middle': 1,
 'F-G3-y|no half': 2*R(27, 5), 'F-W2|12 x 0.5': 12*R(1, 2), 'F-W2|no half': 40*R(1, 2)**2,
 'F-W3|bucket': 50*g98*10, 'F-W3|cable whole': 2*10*g98*10 + 50*g98*10, 'F-W3|cable only': I(2*g98*(10 - y), (y, 0, 10)),
 'V-p11|x-axis': wa(sqrt(x), x**2, 0, 1), 'V-p12|x-axis': wa(-x + 3, x**2 + 1, -2, 1), 'V-p13|x-axis': wa(sqrt(8*x) + 1, x**2 + 1, 0, 2),
 'V-p12|R and r swapped': wa(7 - (-x + 3), 7 - (x**2 + 1), -2, 1),
 'S-val|no r': sh(1, x - x**2, 0, 1), 'S-val|no 2': sh(x, x - x**2, 0, 1)/2,
 'S-indep|h=x': sh(x, x, 0, 1), 'S-indep|area': I(x - x**2, (x, 0, 1)), 'S-indep|bottom minus top': sh(x, x**2 - x, 0, 1), 'S-indep|no 2': sh(x, x - x**2, 0, 1)/2,
 'S-p1|r=x': sh(x, 4*x - x**2 - 3, 1, 3), 'S-p2|r = x - 4': sh(x - 4, 4*x - x**2 - 3, 1, 3), 'S-p3|h top': sh(5 - x, -x + 3, -2, 1), 'S-p4|r=x': sh(x, sqrt(8*x) - x**2, 0, 2),
 'S-p5|no r': sh(1, sin(x), 0, pi), 'S-p5|no 2': sh(x, sin(x), 0, pi)/2, 'S-p6|no r': sh(1, 1 - x**R(1, 4), 0, 1), 'S-p7|no 2': sh(x, sin(x**2), 0, sqrt(pi))/2,
 'S-p8|h top': sh(x, -x**2 + 7*x, R(1, 2), 3), 'S-p9|r=x': sh(x, 4*x - x**2 - 3, 1, 3), 'S-p10|r=x': sh(x, 2*x - x**2, 0, 2),
 'S-m3|no 2': I(2*pi*x*exp(-x**2), (x, 0, sp.oo))/2, 'S-m4|top half': I(2*pi*(u + 3)*sqrt(1 - u**2), (u, -1, 1)),
 'M-S1|r=x': sh(x, x + 2 - x**2, -1, 2), 'M-S1|r = x - 4': sh(x - 4, x + 2 - x**2, -1, 2),
 'M-V4|x-axis': wa(x + 2, x**2, -1, 2), 'M-V4|(R-r)^2': I(pi*((x + 3) - (x**2 + 1))**2, (x, -1, 2)), 'M-S3|no 2': sh(x, exp(x**2), 0, 1)/2,
}
ALIAS = {'S-val': 'S-guided-val'}
MISTV = {lab: fnum(v_) for lab, v_ in MIST.items()}
tolg = lambda w_, k_: w_.get('t', max(6e-4, 1e-4*abs(w_['v'])))   # the grader's matching window for a wrong value
used, bad_w = set(), []
for vid in NEW:
    k_ = KEYS[vid]
    for w_ in k_['wrong']:
        if 'v' not in w_: continue
        hits = [lab for lab, v_ in MISTV.items() if ALIAS.get(lab.split('|')[0], lab.split('|')[0]) == vid and abs(v_ - w_['v']) <= 1e-9*max(1, abs(v_))]
        ans = float(k_['answer'])
        outside = abs(w_['v'] - ans) > max(6e-4, 1e-4*abs(ans)) if not k_.get('tol') else abs(w_['v'] - ans) > float(k_['tol'])
        if len(hits) != 1 or not outside: bad_w.append((vid, w_['v'], hits, outside))
        used |= set(hits)
        print(f"{'OK      ' if len(hits) == 1 and outside else 'PROBLEM '} {vid:<15} shown {w_['v']:<20.13g} = {hits[0].split('|')[1] if hits else '??':<22} message: {w_['m'][:70]}")
unused = sorted(set(MIST) - used)
print(f"\n{len(used)} wrong values matched; problems: {bad_w}; MIST entries not used: {unused}")

OK       V-p11           shown 0.942477796077       = x-axis                 message: That is the answer about the x-axis (3π/10). Measure R and r from the 
OK       V-p12           shown 73.513268094         = x-axis                 message: That is the answer about the x-axis (117π/5). Measure R and r from the
OK       V-p12           shown -124.4070690822      = R and r swapped        message: Negative: R and r are swapped. The axis is above the region, so the lo
OK       V-p13           shown 46.91445029361       = x-axis                 message: That is the answer about the x-axis (224π/15). Measure R and r from th
OK       S-guided-val    shown 1.047197551197       = no r                   message: That is ∫2πh dx, with the radius r = x left out. The integrand is 2π·x
OK       S-guided-val    shown 0.261799387799       = no 2                   message: The 2 in 2π is missing: π∫x(x − x²) dx. A shell’s length is 2πr.
OK       S-indep         shown 2.094395102393       = h=x       

## 5. Numbers and identities quoted in explanations, tryouts, spot-the-error items, figures and "What to notice" lists

Each row: where it appears, the number as shown, an independent SymPy value, the tolerance implied by the rounding shown, and a snippet that must occur in the page source (so the number checked is the number on the page).

In [8]:
Wdown = I(1000*g98*pi*(5*y/12)**2*(12 - y), (y, 0, 12)); Wup = I(1000*g98*pi*(5*(12 - y)/12)**2*(12 - y), (y, 0, 12))
xm8 = pick_root(sp.Eq(R(4, 3)*t**3 + 2*t, 21), t, 0, 3)
CL = [   # (Wolfram label or None, where, shown, SymPy value, tolerance, snippet in the page source)
 ('qK-endpoint average', 'K recognition: endpoint average', 12, (3*0**2 + 3*2**2)/sp.Integer(2)*2, 0, r'\frac{0+12}{2}\cdot2=12'),
 (None, 'K what-to-notice: 1 of 8 units left of x = 1', R(1, 8), I(3*x**2, (x, 0, 1))/8, 0, '(12.5%)'),
 (None, 'K what-to-notice: left half of e^-x', 0.865, 1 - exp(-2), 5e-4, r'$1-e^{-2}\approx0.865$'),
 ('qK-Ex376 total', 'K what-to-notice: total mass of e^-x on [0, 4]', 0.982, 1 - exp(-4), 5e-4, r'of the total $\approx0.982$'),
 (None, 'K what-to-notice: notes p. 9 mass', 103101, R(2, 3)*(E**12 - E**9), 0.5, r'$M\approx103{,}101$'),
 ('qK-e12', 'K what-to-notice: e^12', 163000, E**12, 500, 'about 163,000'),
 ('q-K-p4-mass-ratio-right-left', 'K what-to-notice: 3^x at x = 1 vs x = -1', 9, sp.Integer(3)**1/sp.Integer(3)**-1, 0, 'the right side is $9$ times denser'),
 ('qK-delta(2) x 2', 'K tryout: densest value x length', 24, 3*2**2*2, 0, r'$\delta(2)\cdot2=24$'),
 ('qK-delta(1) x 2', 'K tryout: middle density x length', 6, 3*1**2*2, 0, r'$\delta(1)\cdot2=6$'),
 ('q-K-guided-centroid-x', "K guided step 5: the plate's centroid", 1, I(x*(2*x - x**2), (x, 0, 2))/I(2*x - x**2, (x, 0, 2)), 0, "the plate's centroid is at $x=1$"),
 ('q-K-wrong-limit-xm', 'K spot 2: midpoint with the wrong lower limit', 3.75, pick_root(sp.Eq(R(2, 3)*(sp.exp(3*t) - 1), R(1, 3)*(E**12 - E**9)), t, 0, 5), 5e-3, 'comes out at about 3.75'),
 ('qK-indep area', 'K independent: triangle area', 9, R(1, 2)*3*6, 0, r'$\frac12\cdot3\cdot6=9$'),
 ('qK-p2 area', 'K practice 2: lens area', R(8, 3), I(2 - 2*x**2, (x, -1, 1)), 0, r"The plate's area is $\frac83\approx2.667$"),
 ('qK-p3 left share', 'K practice 3: share of mass in the left half (2026-10-20 wording fix)', 0.88, (1 - exp(-2))/(1 - exp(-4)), 5e-3, 'About 88% of the mass'),
 ('qK-p6 bracket', 'K practice 6: bracket', R(125, 6), R(250, 3) - R(625, 10), 0, r'c\cdot\frac{125}6=25'),
 ('qK-p8 at 9 minus at 3', 'K practice 8: evaluation', 66.044, (R(81, 2)*log(9) - R(81, 4)) - (R(9, 2)*log(3) - R(9, 4)), 5e-4, r'\frac{153}2\ln3-18\approx66.044'),
 ('qK-m3 gap to right end', 'K mastery 3: gap to the right end', 0.69, 10 - log((1 + exp(10))/2), 5e-3, r'$\ln2\approx0.69$'),
 ('qG-seesaw weighted avg', 'G quick check: weighted average', 82.5, R(3*90 + 1*60, 4), 0, r'\frac{3\cdot90+1\cdot60}{4}=82.5'),
 ('q-G-p8-xm', 'G what-to-notice: notes p. 8 mass midpoint', 2.308, xm8, 5e-4, r'$x_m\approx2.308$'),
 (None, 'G guided step 6: half the mass at t', 2.31, xm8, 5e-3, r'$t\approx2.31$'),
 ('qG-p8 density ratio', 'G guided step 5: density ratio', 19, sp.Integer(4*3**2 + 2)/(4*0**2 + 2), 0, '19 times denser'),
 (None, 'G what-to-notice: notes p. 9 x_m and x-bar', 3.719, (11*E**12 - 8*E**9)/(3*(E**12 - E**9)), 5e-4, r'$x_m\approx3.785$ but $\bar x\approx3.719$'),
 ('q-G-385-ratio', 'G what-to-notice: textbook "almost 50%" is really 37%', 1.37, ((1 - 5*exp(-4))/(1 - exp(-4)))/log(2/(1 + exp(-4))), 5e-3, r'$0.925/0.675\approx1.37$'),
 ('qG-p13 density ratio', 'G what-to-notice: 2^x at 3 vs -1', 16, sp.Integer(2)**3/sp.Integer(2)**-1, 0, '16 times larger'),
 (None, 'G what-to-notice / practice 3: notes p. 13 x-bar', 1.5265, g13[1], 5e-5, r'$\bar x\approx1.5265$'),
 (None, 'G practice 3: notes p. 13 y-bar', 4.5339, g13[2], 5e-5, r'$\bar y\approx4.5339$'),
 (None, 'G practice 3: notes p. 13 mass', 25.725, g13[0], 5e-4, r'$M\approx25.725$'),
 (None, 'G what-to-notice: triangle centroid', 2, plate_sym(1, 2*x, 0*x, 0, 3)[2], 0, r'<b>centroid</b> $(2,2)$'),
 ('qG-indep centroid y', 'G independent: constant-density centroid y', R(2, 5), I((x**2 - x**4)/2, (x, 0, 1))/I(x - x**2, (x, 0, 1)), 0, r'$(\frac12,\frac25)$'),
 ('qG-indep Mx', 'G independent: x-bar M', R(11, 60), I(x**2 + x**3 - 2*x**4, (x, 0, 1)), 0, r'=\frac{11}{60}'),
 ('qG-indep My', 'G independent: y-bar M', R(3, 20), R(1, 2)*I(2*x**3 + x**2 - 2*x**5 - x**4, (x, 0, 1)), 0, r'=\frac3{20}'),
 ('qG-spot2 at 4', 'G spot 2: value at x = 4', R(22, 9), R(8, 3) - R(2, 9), 0, r'\frac{22}9e^{12}'),
 ('qG-spot2 at 3', 'G spot 2: value at x = 3', R(16, 9), 2 - R(2, 9), 0, r'\frac{16}9e^9'),
 (None, 'G spot 2: x-bar with the slip, not on the wire', 11.16, AS_WRITTEN/(R(2, 3)*(E**12 - E**9)), 5e-3, r'\approx11.16'),
 ('qG-p6 M', 'G practice 6: mass', 400, I(3*x**2 + 2*x + 4, (x, 2, 7)), 0, '=420-20=400'),
 ('qG-p6 at 7', 'G practice 6: antiderivative at 7', R(25529, 12), R(3, 4)*7**4 + R(2, 3)*7**3 + 2*7**2, 0, r'\frac{25529}{12}-\frac{304}{12}'),
 ('qG-p6 at 2', 'G practice 6: antiderivative at 2', R(304, 12), R(3, 4)*2**4 + R(2, 3)*2**3 + 2*2**2, 0, r'\frac{25529}{12}-\frac{304}{12}'),
 ('qG-p7 Mx', 'G practice 7: x-bar M', R(14, 15), I(4*x**2 - 2*x**4, (x, 0, 1)), 0, r'\frac43-\frac25=\frac{14}{15}'),
 ('qG-p8 My', 'G practice 8: y-bar M', R(9, 2), R(1, 2)*I(x**5 - x, (x, 1, 2)), 0, r'\bar yM=\frac12\int_1^2(x^5-x)\,dx=\frac92'),
 ('qG-p9 Mx', 'G practice 9: x-bar times area', R(1, 15), I(x*(sqrt(x) - x), (x, 0, 1)), 0, r'\frac25-\frac13=\frac1{15}'),
 (None, 'G practice 9: sqrt(0.4)', 0.63, sqrt(R(2, 5)), 5e-3, r'\sqrt{0.4}\approx0.63'),
 ('qG-m4 integral', 'G mastery 4: integral', R(128, 3), I((16 - x**2)/2, (x, -4, 4)), 0, r'=\frac{128}3'),
 ('qG-F-G3 integral', 'final 39: integral', R(1944, 5), I(81 - x**4, (x, -3, 3)), 0, r'=\frac{1944}5'),
 ('qW-slab weight', 'W quick check: slab weight', 24500, 1000*g98*25*R(1, 10), 0, r'$24{,}500$'),
 ('q-W-p17-weight', 'W center-of-mass check: water weight', 2205000, 1000*g98*225, 0, r'1000\cdot9.8\cdot225=2{,}205{,}000'),
 ('qW-p17 cm check', 'W center-of-mass check: weight x 9.5 m', 20947500, 1000*g98*225*(14 - R(9, 2)), 0, r'$2{,}205{,}000\times9.5=20{,}947{,}500$'),
 ('qW-guided bracket', 'W guided step 5: bracket', 1728, I(12*y**2 - y**3, (y, 0, 12)), 0, r'\big(6912-5184\big)'),
 ('q-W-p18-weight', 'W guided step 6: cone weight', 980000*pi, 9800*R(1, 3)*pi*25*12, 0, r'=980{,}000\pi$ N'),
 ('q-W-p18-cm-height', 'W guided step 6: cone center of mass height', 9, I(y*pi*(5*y/12)**2, (y, 0, 12))/I(pi*(5*y/12)**2, (y, 0, 12)), 0, 'at $y=9$'),
 ('qW-cone share', 'W what-to-notice: a quarter of weight x 12', R(1, 4), Wdown/(1000*g98*pi*R(1, 3)*25*12*12), 0, 'only a quarter of'),
 (None, 'W what-to-notice: about 9.24 million J', 9.24e6, Wdown, 5e3, r'\approx9.24$ million J'),
 (None, 'W tryout 3: limits 0..12', 23520000, I(1000*25*g98*(14 - y), (y, 0, 12)), 0, r'$23{,}520{,}000$ J instead of'),
 ('qW-indep cm check', 'W independent: weight x 25 ft', 2496000, R(624, 10)*160*10*(30 - 5), 0, r'$99{,}840\times25=2{,}496{,}000$'),
 ('qW-p2 bracket', 'W practice 2: arithmetic', 20947500, 245000*(14*9 - R(81, 2)), 0, r'245{,}000\cdot85.5=20{,}947{,}500'),
 ('qW-p3 by u', 'W practice 3: by substitution', 74880000*pi, 3*pi*I(u**3, (u, 20, 100)), 0, r'\frac{3\pi}4\big(100^4-20^4\big)=74{,}880{,}000\pi'),
 ('qW-spring triangle', 'W what-to-notice: triangle area', R(15, 4), R(1, 2)*R(1, 2)*15, 0, r'\frac12\cdot\frac12\cdot15=\frac{15}4'),
 ('qW-bucket trapezoid', 'W what-to-notice: bucket trapezoid', 2700, R(32 + 22, 2)*100, 0, r'\frac{32+22}2\times100=2700'),
 ('qW-p9 per-foot leak', 'W practice 9 wrong message: leak per foot', 0.125, R(1, 2)/4, 0, '0.125 lb per foot'),
 ('qW-m3 height then', 'W mastery 3: height when 1 lb is left', 98, 2*sp.solve(sp.Eq(50 - t, 1), t)[0], 0, r'$y=98$ ft'),
 (None, 'W mastery 3: pieces', 2521, (4900 - 2401) + 22, 0, r'\big(4900-2401\big)+22=2521'),
 (None, 'W mastery 2: vertex-up cone takes three times the work', 3, Wup/Wdown, 0, 'three times the work'),
 ('qW-M-W2 k', 'Mixed 34: spring constant', 300, 6/(R(1, 2)*R(1, 5)**2), 0, r'$k=300$ N/m'),
 (None, 'Mixed 34: doubling the stretch quadruples the work', 24, I(300*x, (x, 0, R(2, 5))), 0, 'quadruples the work (24 J from 0)'),
 (None, 'Mixed 32: arithmetic', 490000*pi, 39200*pi*R(25, 2), 0, r'39{,}200\pi\cdot12.5=490{,}000\pi'),
 ('q-W-315-cm', 'W explorer formula: textbook Ex. 3.9.15', 3136566, 998400*pi, 0.5, r'998{,}400\pi\approx3{,}136{,}566'),
 ('qS-p1 washer route', 'S practice 1: check by washers in y', 8*pi/3, 4*pi*I(sqrt(4 - y), (y, 3, 4)), 0, r'4\pi\int_3^4\sqrt{4-y}\,dy=\frac{8\pi}3'),
 ('qS-p6 washer route', 'S practice 6: check by washers', pi/9, pi*I(y**8, (y, 0, 1)), 0, r'\pi\int_0^1y^8\,dy=\frac\pi9'),
 ('qS-guided cylinder bound', 'S guided step 7: enclosing cylinder', pi/4, pi*1**2*R(1, 4), 0, r'\pi(1)^2\cdot\frac14=\frac\pi4\approx0.785'),
 ('qV-p11 shells in y', 'V practice 11: check by shells in y', 49*pi/30, 2*pi*I((y + 2)*(sqrt(y) - y**2), (y, 0, 1)), 0, r'2\pi\int_0^1(y+2)(\sqrt y-y^2)\,dy=\frac{49\pi}{30}'),
 ('qS-m4 doughnut rule', 'S mastery 4: doughnut rule', 6*pi**2, pi*1**2*2*pi*3, 0, r'\pi(1)^2\cdot2\pi(3)=6\pi^2'),
 ('qS-p5 by parts', 'S practice 5: by parts', pi, I(x*sin(x), (x, 0, pi)), 0, r'\big[-x\cos x+\sin x\big]_0^\pi=\pi'),
 (None, 'S practice 9: 8 pi/3 per unit of distance from x = 2', 8*pi/3, (sh(5 - x, 4*x - x**2 - 3, 1, 3) - sh(4 - x, 4*x - x**2 - 3, 1, 3)), 0, r'$\frac{8\pi}{3}$ per unit of distance'),
]
bad_cl = []
for lab, where, shown, val, tol, snip in CL:
    v_ = fnum(val); okv = abs(fnum(shown) - v_) <= max(tol, 1e-12*max(1, abs(v_))); oks = snip in PARTS
    if not (okv and oks): bad_cl.append((where, shown, v_, okv, oks))
    print(f"{'OK      ' if okv and oks else 'PROBLEM '} {where:<62.62} shown {str(shown):<14.14} value {v_:<20.12g} on the page: {oks}")
# the 2026-10-20 bucket formula, symbolically
nn = sp.Symbol('n', positive=True, integer=True); jj = sp.Symbol('j', integer=True)
bucket_n = sp.simplify(sp.summation((32 - R(1, 10)*jj*(100/nn))*(100/nn), (jj, 1, nn)))
bucket_ok = sp.simplify(bucket_n - (2700 - 500/nn)) == 0 and r'2700-\frac{500}n' in PARTS
print(f"W what-to-notice: right-end bars for the leaky bucket sum to {bucket_n}  (the page says 2700 - 500/n): {bucket_ok}")
print(f"\n{len(CL)} quoted numbers; problems: {bad_cl}")

OK       K recognition: endpoint average                                shown 12             value 12                   on the page: True
OK       K what-to-notice: 1 of 8 units left of x = 1                   shown 1/8            value 0.125                on the page: True
OK       K what-to-notice: left half of e^-x                            shown 0.865          value 0.864664716763       on the page: True
OK       K what-to-notice: total mass of e^-x on [0, 4]                 shown 0.982          value 0.981684361111       on the page: True
OK       K what-to-notice: notes p. 9 mass                              shown 103101         value 103101.138328        on the page: True
OK       K what-to-notice: e^12                                         shown 163000         value 162754.791419        on the page: True
OK       K what-to-notice: 3^x at x = 1 vs x = -1                       shown 9              value 9                    on the page: True
OK       K tryout: densest value x

## 6. Explorer reference values

The browser test (`test_guide.py`) compares the live numbers of the shells (S), mass (K), balance (G) and work (W) explorers with independent mpmath quadrature on hundreds of preset/parameter combinations. This section checks the closed-form constants built into `app.js`.
- **Shells explorer.** It computes $V(c)=2\pi\,|J_1-c\,J_0|$ with $J_0=\int(\text{up}-\text{lo})\,dx$ and $J_1=\int x(\text{up}-\text{lo})\,dx$, for any vertical axis $x=c$ outside the region. That is the shell integral rewritten (it is only used inside the explorer, not taught); here it is compared with direct quadrature of $2\pi|x-c|\,h(x)$.
- **Mass and balance explorers.** Their preset masses, moments and mass midpoints are written as closed forms; each must occur in `app.js` and equal the SymPy value.

In [9]:
APP = (BUILD / 'app.js').read_text(encoding='utf-8')
SPRE = {  # preset: (up, lo, a, b, J0, J1, app.js text)
 'MATH1342 p. 5':  (x - x**2, 0*x, 0, 1, R(1, 6), R(1, 12), 'J0: 1 / 6, J1: 1 / 12'),
 'MATH1342 p. 7':  (4*x - x**2, 3 + 0*x, 1, 3, R(4, 3), R(8, 3), 'J0: 4 / 3, J1: 8 / 3'),
 'MATH1342 p. 8':  (-x + 3, x**2 + 1, -2, 1, R(9, 2), -R(9, 4), 'J0: 9 / 2, J1: -9 / 4'),
 'MATH1342 p. 9':  (sqrt(8*x) + 1, x**2 + 1, 0, 2, R(8, 3), R(12, 5), 'J0: 8 / 3, J1: 12 / 5'),
 'Ex. 3.5.12':     (2 - x, sqrt(x), 0, 1, R(5, 6), R(4, 15), 'J0: 5 / 6, J1: 4 / 15'),
}
EXPL = {}
worst_s, nS, sym_s = 0.0, 0, True
for key, (up_, lo_, a, b, J0, J1, txt) in SPRE.items():
    sym_s &= sp.simplify(I(up_ - lo_, (x, a, b)) - J0) == 0 and sp.simplify(I(x*(up_ - lo_), (x, a, b)) - J1) == 0 and txt in APP
    fu, fl = sp.lambdify(x, up_, 'mpmath'), sp.lambdify(x, lo_, 'mpmath')
    for cc in [a - 3, a - 1.5, a - 0.5, a, b, b + 0.5, b + 2, b + 4]:
        direct = mq(lambda s: 2*mp.pi*abs(s - cc)*(fu(s) - fl(s)), [a, (a + b)/2, b])
        worst_s = max(worst_s, abs(float(direct) - float(2*sp.pi*abs(J1 - sp.nsimplify(cc)*J0)))); nS += 1
    EXPL['xS-J0 ' + {'MATH1342 p. 5': 'notes p5', 'MATH1342 p. 7': 'notes p7', 'MATH1342 p. 8': 'notes p8', 'MATH1342 p. 9': 'notes p9', 'Ex. 3.5.12': 'Ex3512'}[key]] = J0
    EXPL['xS-J1 ' + {'MATH1342 p. 5': 'notes p5', 'MATH1342 p. 7': 'notes p7', 'MATH1342 p. 8': 'notes p8', 'MATH1342 p. 9': 'notes p9', 'Ex. 3.5.12': 'Ex3512'}[key]] = J1
EXPL['xS-V Ex3512'] = sh(x, 2 - x - sqrt(x), 0, 1)
print(f"shells explorer: J0, J1 match SymPy and app.js: {sym_s}; 2 pi |J1 - c J0| vs direct quadrature on {nS} preset/axis combinations, worst difference {worst_s:.2e}")
KG = [   # (app.js text, [(hand value, SymPy value)])
 ("M: 8, xm: Math.cbrt(4)", [(8, I(3*x**2, (x, 0, 2))), (sp.cbrt(4), mid_sym(3*x**2, 0, 2))]),
 ("M: 1 - Math.exp(-4), xm: Math.log(2 / (1 + Math.exp(-4)))", [(1 - exp(-4), I(exp(-x), (x, 0, 4))), (log(2/(1 + exp(-4))), mid_sym(exp(-x), 0, 4))]),
 ("M: 2 / 3 * (Math.exp(12) - Math.exp(9)), xm: Math.log((Math.exp(12) + Math.exp(9)) / 2) / 3", [(R(2, 3)*(E**12 - E**9), I(d9, (x, 3, 4))), (log((E**12 + E**9)/2)/3, mid_sym(d9, 3, 4))]),
 ("M: 4 / 3, u: ''", [(R(4, 3), plate_sym(x, x - x**2, -x, 0, 2)[0])]),
 ("M: 8 * (5 * L3 - 4) / (3 * L3 * L3 * L3)", [(8*(5*log(3) - 4)/(3*log(3)**3), plate_sym(3**x, 2 - x**2, x**2, -1, 1)[0])]),
 ("M: 45 / 2, u: 'kg'", [(R(45, 2), plate_sym(x**2 + 1, 0*x, 2*x - 6, 0, 3)[0])]),
 ("M: 90, Mx: 180", [(90, 30 + 60), (180, 30*0 + 60*3)]),
 ("M: 42, Mx: 90", [(42, gm[0]), (90, I(x*(4*x**2 + 2), (x, 0, 3)))]),
 ("M: 2 / 3 * (E(12) - E(9)), Mx: 2 / 9 * (11 * E(12) - 8 * E(9))", [(R(2, 3)*(E**12 - E**9), I(d9, (x, 3, 4))), (R(2, 9)*(11*E**12 - 8*E**9), I(x*d9, (x, 3, 4)))]),
 ("M: 1 - E(-4), Mx: 1 - 5 * E(-4)", [(1 - exp(-4), I(exp(-x), (x, 0, 4))), (1 - 5*exp(-4), I(x*exp(-x), (x, 0, 4)))]),
 ("M: 2, Mx: 3, My: 1", [(2, g12[0]), (3, g12[0]*g12[1]), (1, g12[0]*g12[2])]),
 ("M: 9 / 4, Mx: 58 / 15, My: 9 / 2", [(R(9, 4), g8[0]), (R(58, 15), g8[0]*g8[1]), (R(9, 2), g8[0]*g8[2])]),
 ("M: 9, Mx: 18, My: 18", [(9, I(2*x, (x, 0, 3))), (18, I(x*2*x, (x, 0, 3))), (18, I(x*2*x, (x, 0, 3)))]),
]
kg_ok = True
for txt, pairs in KG:
    ok_ = txt in APP and all(abs(fnum(h_) - fnum(s_)) <= 1e-12*max(1, abs(fnum(h_))) for h_, s_ in pairs); kg_ok &= ok_
    print(f"{'OK      ' if ok_ else 'PROBLEM '} app.js: {txt}")
EXPL.update({'xG-M Ex385': 1 - exp(-4), 'xG-Mx Ex385': 1 - 5*exp(-4), 'xG-M Ex387': g8[0], 'xG-Mx Ex387': g8[0]*g8[1], 'xG-My Ex387': g8[0]*g8[2],
             'xG-M Ex388': 9, 'xG-Mx Ex388': 18, 'xG-My Ex388': 18, 'xK-M p4 widget form': 8*(5*log(3) - 4)/(3*log(3)**3), 'xW-bucket sum n=10': 2650})
print("mass and balance explorer constants all present and correct:", kg_ok)

shells explorer: J0, J1 match SymPy and app.js: True; 2 pi |J1 - c J0| vs direct quadrature on 40 preset/axis combinations, worst difference 3.55e-15


OK       app.js: M: 8, xm: Math.cbrt(4)
OK       app.js: M: 1 - Math.exp(-4), xm: Math.log(2 / (1 + Math.exp(-4)))
OK       app.js: M: 2 / 3 * (Math.exp(12) - Math.exp(9)), xm: Math.log((Math.exp(12) + Math.exp(9)) / 2) / 3
OK       app.js: M: 4 / 3, u: ''
OK       app.js: M: 8 * (5 * L3 - 4) / (3 * L3 * L3 * L3)
OK       app.js: M: 45 / 2, u: 'kg'
OK       app.js: M: 90, Mx: 180
OK       app.js: M: 42, Mx: 90
OK       app.js: M: 2 / 3 * (E(12) - E(9)), Mx: 2 / 9 * (11 * E(12) - 8 * E(9))
OK       app.js: M: 1 - E(-4), Mx: 1 - 5 * E(-4)
OK       app.js: M: 2, Mx: 3, My: 1
OK       app.js: M: 9 / 4, Mx: 58 / 15, My: 9 / 2
OK       app.js: M: 9, Mx: 18, My: 18
mass and balance explorer constants all present and correct: True


## 7. Wolfram Language cross-check

`wolfram_shells_mass_work_checks.txt` holds six batches, inputs and outputs verbatim, computed by Wolfram Language from the problem statements, not from the answers:
1. every new §3.7 and §3.8 answer; 2. every new §3.9, mixed and final answer, plus the 2026-10-14 shells and washer answers;
3. the targeted wrong-answer values; 4. the lecture's worked answers, the p. 9 slip, and quoted numbers;
5a. (2026-10-20) the wrong values and intersection limits batch 3 left out, the explorers' closed forms, more quoted numbers; 5b. the typed expressions.

Every row must match a value verified above, and every verified answer must have a Wolfram row.

In [10]:
WTXT = (VER / 'wolfram_shells_mass_work_checks.txt').read_text(encoding='utf-8')
chunks = [b_.split("\n" + "=" * 100)[0] for b_ in re.split(r"-{40} OUTPUT", WTXT)[1:]]
def parse_out(s_):
    body = re.search(r'Out\[\d+\]= "(.*)"', s_, re.S).group(1); rows = {}
    for line in body.split("\\n"):
        parts = [p_.strip() for p_ in line.split(" | ")]
        if len(parts) == 3 and parts[1].startswith('key = intended'): parts = [parts[0], parts[1], parts[2]]
        rows[parts[0]] = parts[1:]
    return rows
B = [parse_out(ch) for ch in chunks]
print("rows per batch:", [len(b_) for b_ in B])
def wnum(s_):
    mant = s_.split('`')[0]; e_ = re.search(r'\*\^(-?\d+)', s_); return float(mant) * (10 ** int(e_.group(1)) if e_ else 1.0)
CLW = {lab: fnum(val) for lab, where, shown, val, tol, snip in CL if lab}
SPECIAL = {'n37-p9-Mx-as-written': fnum(AS_WRITTEN), 'n37-p9-antideriv-check': 0.0,
           'q-W-up-cone-full': fnum(Wup), 'qW-p17 weight': fnum(1000*g98*225), 'qW-m3 time to 1 lb': 49.0,
           'q-W-316': fnum(I(R(624, 10)*16*pi*y**2/225*(30 - y), (y, 0, 10)))}   # textbook Ex. 3.9.16 (quoted in NOTES_INDEX)
WOLFRAM, dis, unmatched = {}, [], []
for bi, rows in enumerate(B[:5], start=1):
    for lab, (ex, nu) in rows.items():
        if lab == 'xW-bucket sum n':
            if ex.replace(' ', '') != '2700-500/n': dis.append((lab, ex)); continue
            continue
        w_ = wnum(nu)
        ref = VERIFIED.get(lab, MISTV.get(lab, CLW.get(lab, SPECIAL.get(lab, fnum(EXPL[lab]) if lab in EXPL else None))))
        if ref is None: unmatched.append(lab); continue
        if abs(w_ - ref) > 1e-11 * max(1, abs(ref)): dis.append((lab, ex, w_, ref))
        WOLFRAM[lab] = (ex, w_)
expr_rows = B[5]
expr_ok = sorted(expr_rows) == sorted(EXPR_REF) and all(v_[0].endswith('True') and v_[1].endswith('True') for v_ in expr_rows.values())
answers_needing = [v for v in NEW if KEYS[v]['kind'] == 'num']
no_wolf = [v for v in answers_needing if v not in WOLFRAM]
mist_no_wolf = [m_ for m_ in used if m_ not in WOLFRAM]
print(f"Wolfram rows compared: {len(WOLFRAM)} | disagreements: {dis} | rows with nothing to compare against: {unmatched}")
print(f"new numeric answers without a Wolfram row: {no_wolf} | wrong values without a Wolfram row: {mist_no_wolf}")
print(f"typed expressions (batch 5b): {len(expr_rows)} rows, all key = intended and every wrong differs: {expr_ok}")
print("sample:", {k_: WOLFRAM[k_][0] for k_ in ['K-p2-a', 'G-p1-x', 'W-p3', 'S-m4', 'n37-p13-ybar'] if k_ in WOLFRAM})

rows per batch: [48, 53, 87, 41, 100, 29]
Wolfram rows compared: 328 | disagreements: [] | rows with nothing to compare against: []
new numeric answers without a Wolfram row: [] | wrong values without a Wolfram row: []
typed expressions (batch 5b): 29 rows, all key = intended and every wrong differs: True
sample: {'K-p2-a': '-1', 'G-p1-x': '(8 - 11*E^3)/(3 - 3*E^3)', 'W-p3': '74880000*Pi', 'S-m4': '6*Pi^2', 'n37-p13-ybar': '(-90 + Log[2]*(294 + Log[2]*(-399 + 290*Log[2])))/(Log[2]^2*(-15 + 34*Log[2]))'}


## 8. Final pass on the built guide

- Every embedded answer that is new since 2026-10-14 must be verified above (SymPy/mpmath, the guide's grader, and Wolfram).
- Every answer that existed before 2026-10-14 must be unchanged, and so must every 2026-10-14 answer since 2026-10-19.
- No interactive item may be lost (ids compared with both snapshots).

In [11]:
src = GUIDE.read_text(encoding='utf-8')
inputs = re.findall(r"""<input class="ans"((?:[^>"']|"[^"]*"|'[^']*')*)>""", src)
bad_ans, n_old, n_14, n_19 = [], 0, 0, 0
for attrs in inputs:
    vid = re.search(r'data-vid="([^"]+)"', attrs).group(1); ans = re.search(r'data-answer="([^"]+)"', attrs).group(1)
    kind = (re.search(r'data-kind="([^"]+)"', attrs) or [None, 'num'])[1]
    if vid in b14:
        n_old += 1
        if b14[vid]['answer'] != ans: bad_ans.append((vid, 'CHANGED since 10-14', b14[vid]['answer'], ans))
        continue
    if vid in b19:
        n_14 += 1
        if b19[vid]['answer'] != ans: bad_ans.append((vid, 'CHANGED since 10-19', b19[vid]['answer'], ans))
    else: n_19 += 1
    if kind in ('anti', 'expr'):
        if not EXPR_OK.get(vid) or vid not in expr_rows: bad_ans.append((vid, 'expression not verified'))
        continue
    ref = VERIFIED.get(vid)
    if ref is None: bad_ans.append((vid, 'NOT VERIFIED')); continue
    if abs(float(ans) - ref) > 1e-9 * max(1, abs(ref)): bad_ans.append((vid, ans, ref))
    if vid not in WOLFRAM: bad_ans.append((vid, 'no Wolfram check'))
now_vids = {re.search(r'data-vid="([^"]+)"', a_).group(1) for a_ in inputs}
body = re.sub(r'<(script|style)[\s\S]*?</\1>', ' ', src)
ids_now = set(re.findall(r'\s(?:data-pid|data-qid|data-gid|data-sid|data-eid|id)="([^"]+)"', body))
lost = sorted((set(b14) | set(b19)) - now_vids); lost_ids = sorted(set(BEFORE19['ids']) - ids_now)
old_ok = all(b14[v]['answer'] == b19[v]['answer'] for v in b14)
counts = {
 "answer_inputs": len(inputs), "problems": len(re.findall(r'<div class="problem"', src)), "mcq": len(re.findall(r'<div class="mcq"', src)),
 "spots": len(re.findall(r'<div class="spot"', src)), "tryouts": len(re.findall(r'<div class="tryout"', src)), "guided": len(re.findall(r'<div class="guided"', src)),
 "explain": len(re.findall(r'<div class="explain card"', src)), "widgets": len(re.findall(r'<div class="widget"', src)), "topic_sections": len(re.findall(r'<section id="[A-Z]" class="topic"', src))}
print(f"answers embedded: {len(inputs)} = {n_old} from before 2026-10-14 (unchanged) + {n_14} from 2026-10-14 + {n_19} from 2026-10-19 | problems: {bad_ans}")
print("lost answers:", lost, "| lost ids:", lost_ids, "| pre-10-14 answers also unchanged in the 10-19 snapshot:", old_ok)
print("counts 2026-10-14 -> 2026-10-19 -> now:"); [print(f"   {k_:<15} {BEFORE14['counts'][k_]:>4} -> {BEFORE19['counts'][k_]:>4} -> {v_:>4}") for k_, v_ in counts.items()]

answers embedded: 354 = 219 from before 2026-10-14 (unchanged) + 42 from 2026-10-14 + 93 from 2026-10-19 | problems: []
lost answers: [] | lost ids: [] | pre-10-14 answers also unchanged in the 10-19 snapshot: True
counts 2026-10-14 -> 2026-10-19 -> now:
   answer_inputs    219 ->  261 ->  354
   problems         162 ->  181 ->  231
   mcq               97 ->  105 ->  130
   spots             18 ->   20 ->   26
   tryouts           13 ->   16 ->   25
   guided            10 ->   11 ->   14
   explain           11 ->   12 ->   15
   widgets           10 ->   11 ->   14
   topic_sections    10 ->   11 ->   14


[None, None, None, None, None, None, None, None, None]

## 9. Scope checks

- The new sections may mention excluded material (3D solid densities, a density depending on $y$, potential energy, changing-mass mechanics, hydrostatic pressure, surface area, the cone's centroid) only to say it isn't in the notes, as a reference, or as a wrong option.
- No new §1.1/§1.3 items: the counts of items tagged U, P, R must be unchanged (the open question from 2026-09-25 is untouched).

In [12]:
vis = re.sub(r'<annotation[\s\S]*?</annotation>', ' ', body)
def section_text(sid):
    m_ = re.search(r'<section id="%s" class="topic">([\s\S]*?)</section>' % sid, vis)
    return re.sub(r'\s+', ' ', _html.unescape(re.sub(r'<[^>]+>', ' ', m_.group(1)))) if m_ else ''
newtxt = ' '.join(section_text(s_) for s_ in 'KGW')
for term in ['potential energy', 'slug', 'hydrostatic', 'surface area', 'Example 3.8.9', 'Example 3.7.8', 'depends on y', 'solid', 'Newton']:
    hits = [m_.start() for m_ in re.finditer(re.escape(term), newtxt, re.I)]
    print(f"   '{term}': {len(hits)} mention(s)", [newtxt[max(0, h_ - 60): h_ + 30] for h_ in hits][:3])
print("topic sections:", re.findall(r'<section id="([A-Z])" class="topic"', src))
upr = {T_: (BEFORE14['topic_attr_counts'][T_], BEFORE19['topic_attr_counts'][T_], len(re.findall(r'data-topic="%s"' % T_, src))) for T_ in 'UPR'}
print("items tagged with the §1.1/§1.3 topics, 10-14 -> 10-19 -> now:", upr)
scope_ok = all(a_ == b_ == c_ for a_, b_, c_ in upr.values())
print("items per new topic:", {T_: len(re.findall(r'data-topic="%s"' % T_, src)) for T_ in 'SKGW'})

   'potential energy': 0 mention(s) []
   'slug': 8 mention(s) [' a b \u200b δ ( x ) d x density : mass per unit length (kg/m, or slugs/ft) at the point x. It i', 'imits and stop (notes p. 4). The problem gives units (kg/m, slugs/ft, kg/m²) Your answer g', 'its (kg/m, slugs/ft, kg/m²) Your answer gets units too: kg, slugs (notes p. 1, green). Tem']
   'hydrostatic': 0 mention(s) []
   'surface area': 0 mention(s) []
   'Example 3.8.9': 1 mention(s) ['f mass is 3 4 4 3 \u200b of the way up from the vertex (textbook Example 3.8.9), at y = 9 y = 9']
   'Example 3.7.8': 0 mention(s) []
   'depends on y': 0 mention(s) []
   'solid': 0 mention(s) []
   'Newton': 6 mention(s) ['heck: lift 20 lb through 3 ft. show 60 60 ft-lb. In metric, newtons × metres = joules. Wei', 'density 1000 kg/m³ is a mass per volume, so to get a force (newtons) you multiply by g = 9', "rce. No conversion needed. 9.8 m/s² turns a mass in kg into newtons. It's for kg/m³ (notes"]
topic sections: ['U', 'P', 'R', 'A', 'B

## 10. Summary

In [13]:
bad_rows = [r_ for r_ in ROWS if not r_['ok']]
print(f"SymPy/mpmath checks: {len(ROWS)} rows, {len(bad_rows)} mismatches")
print(f"typed expressions: {sum(EXPR_OK.values())}/{len(EXPR_OK)} | the guide's grader on every key: {js_ok} | targeted wrong values: {len(used)} | quoted numbers: {len(CL)}")
print(f"Wolfram: {len(WOLFRAM)} rows compared, {len(dis)} disagreements, {len(unmatched)} unmatched; typed expressions {expr_ok}")
print(f"built guide: {len(inputs)} answers ({n_14 + n_19} new since 2026-10-14, all verified: {not bad_ans}); nothing lost: {not lost and not lost_ids}; older answers unchanged: {old_ok}; scope: {scope_ok}")
assert not bad_rows and not bad_ans and not lost and not lost_ids and old_ok and js_ok and all(EXPR_OK.values()) and worst_js < 1e-9
assert not bad_w and not unused and not bad_cl and bucket_ok and sym_s and worst_s < 1e-12 and kg_ok
assert not dis and not unmatched and not no_wolf and not mist_no_wolf and expr_ok and scope_ok
json.dump(VERIFIED, open(VER / 'verified_values_shells_mass_work.json', 'w'), indent=1)
print("all checks passed; saved verified_values_shells_mass_work.json")

SymPy/mpmath checks: 134 rows, 0 mismatches
typed expressions: 29/29 | the guide's grader on every key: True | targeted wrong values: 113 | quoted numbers: 72
Wolfram: 328 rows compared, 0 disagreements, 0 unmatched; typed expressions True
built guide: 354 answers (135 new since 2026-10-14, all verified: True); nothing lost: True; older answers unchanged: True; scope: True
all checks passed; saved verified_values_shells_mass_work.json
